# Attempt Day 1

based on session notes, the defined classes and excution happy paths. are the core outer requirements to build

In [1]:
class ParkingSpot:
    def __init__(self):
        self.occupant = None

    # [2] Mutator for spot occupancy. Local enforcement point: only assign when the spot
    # is empty and the vehicle type is compatible with this spot.
    def assign(self, vehicle):
        if self.occupant is not None: 
            return False
        else: #is None, hence self.occupant can be set to vehicle.
            self.occupant = vehicle
            return True
    
    # check unpark for later.
    def release(self):
        if self.occupant is None: 
            return False
        else: 
            self.occupant = None
            return True
    def get_occupant(self):
        return self.occupant

# TODO: design vehicle
class Vehicle:
    def __init__(self):
        pass

# TODO: design Parking Ticket.

class ParkingTicket:
    def __init__(self):
        pass

class ParkingFloor:
    def __init__(self,size):
        self.parking_grid = [ParkingSpot() for _ in range(self.size)]
        self.size = size
        self.filled = 0

    def full(self):
        if self.size == self.filled:
            return True
        else:
            return False

    # TODO: Move controller pattern from ParkingFloor to ParkingLot
    # TODO:
    def park(self, vehicle):
        if not self.full():
            #naive parkingspot implementation:
            for parking_spot in self.parking_grid:
                if parking_spot.assign(vehicle):
                    break
            self.filled += 1
        
    def unpark(self, vehicle):
        for parking_spot in self.parking_grid:
            if parking_spot.get_occupant() == vehicle and parking_spot.release(vehicle):
                self.filled -= 1
    
class ParkingLot:
    def __init__(self):
        pass
    def park(self, vehicle):
        pass
    def unpark(self,vehicle):
        pass


### Critique After Attempt 1 (Hard, no spoilers)

Reference: `../2026-05-27-session.md` sections **2A, 2E, 4A, 5A**.

- `High:` You coded classes before locking invariants/state machine. This repeats the workflow skip called out in notes (`2B`, `5A`): you are still describing behavior, not proving correctness constraints.
- `High:` Ownership is split wrong: `ParkingFloor.park/unpark` still mutates core lifecycle while your own notes converged that lot-level transitions must be owned by `ParkingLot` (`4C`: mutation closure).
- `High:` `Ticket` is structurally empty for exit semantics. Notes explicitly challenged this (`2E`, `2G`): unpark keyed by session handle is impossible to guarantee with current model.
- `Medium:` `ParkingSpot` has occupancy but no compatibility state, despite notes saying compatibility is a first-class requirement (`1B`, `2A`, `2E`).
- `Medium:` There is no invariant guard for double-booking across the system boundary; local `assign` checks are not enough as a design claim (`5A`, `5C`).

Verdict: this attempt is implementation-first and still under-modeled at the LLD level.


## Attempt 1 Continued but refactoring

1. It seems that ParkingLot is an aggregate root since it owns one to many ParkingFloors and ParkingFloors own ParkingSpots.
2. ParkingSpot should track both compatibility and occupancy, not only which vehicle is currently there.
3. Ticket likely needs more than entry time. At minimum, think about whether the system can unpark correctly if the ticket does not identify the parked vehicle or spot.
4. ParkingLot and ParkingFloor should just be collections instead of state tracking from previous
5. Move parking controller logic to ParkingLot as the core aggregate

Run with naive search and park unpark implemenation for now.

In [ ]:
class ParkingSpot:
    def __init__(self):
        self.occupant = None

    # [2] Mutator for spot occupancy. Local enforcement point: only assign when the spot
    # is empty and the vehicle type is compatible with this spot.
    def assign(self, vehicle):
        if self.occupant is not None: 
            return False
        else: #is None, hence self.occupant can be set to vehicle.
            self.occupant = vehicle
            return True
    
    # check unpark for later.
    def release(self):
        if self.occupant is None: 
            return False
        else: 
            self.occupant = None
            return True
    def get_occupant(self):
        return self.occupant

# TODO: design vehicle
class Vehicle:
    def __init__(self, id = None, type = None, ticket = None):
        self.id = id
        self.type = type
        self.ticket = ticket
        

# TODO: design Parking Ticket.

class ParkingTicket:
    def __init__(self, id = None, entry_time = None):
        self.id = id
        self.entry_time = entry_time

class ParkingFloor:
    def __init__(self,size):
        self.parking_grid = [ParkingSpot() for _ in range(self.size)]
        self.size = size
        self.filled = 0

    def full(self):
        if self.size == self.filled:
            return True
        else:
            return False

    # TODO: Move controller pattern from ParkingFloor to ParkingLot
    # TODO:
    def park(self, vehicle):
        if not self.full():
            #naive parkingspot implementation:
            for parking_spot in self.parking_grid:
                if parking_spot.assign(vehicle):
                    break
            self.filled += 1
        
    def unpark(self, vehicle):
        for parking_spot in self.parking_grid:
            if parking_spot.get_occupant() == vehicle and parking_spot.release(vehicle):
                self.filled -= 1
    

class ParkingLot:
    def __init__(self, parking_floors = 10, parking_floor_size = 10):
        self.parking_floors = [ParkingFloor(size = parking_floor_size)]
    def park(self, vehicle):
        parked = False
        for parking_floor in self.parking_floors:
            if parking_floor.park(vehicle):
                parked = True
                break
        return parked

    def unpark(self,vehicle):
        unparked = False
        for parking_floor in self.parking_floors:
            if parking_floor.unpark(vehicle):
                unparked = True
                break
        return unparked


This is a great question because it gets to the heart of **API design, DDD aggregates, identity, and state transitions**.

Let's analyze it from first principles.

---

# 1. The core operation is a state transition

The parking lot state can be modeled as:

[
ParkingLotState = { Spots, Vehicles, Tickets }
]

The two transitions are:

```
park(vehicle)   : State → State
unpark(ticket)  : State → State
```

The question is:

> What information is the minimal, stable, authoritative identity needed to perform each transition?

---

# 2. Why park takes a Vehicle

## Intent

The user's intent is:

> "I have a vehicle. Please find a place for it."

Before parking, the vehicle has no relationship with the parking lot.

The system needs information from the vehicle:

```python
Vehicle {
    id
    type  # car, motorcycle, truck
}
```

The algorithm is:

```
park(vehicle):
    spot = find_available_spot(vehicle.type)
    occupy(spot, vehicle)
    ticket = issue_ticket(vehicle, spot, entry_time)
    return ticket
```

The **vehicle determines the constraints**.

The parking lot is solving a search problem:

```
Vehicle type
      |
      v
Find compatible spot
      |
      v
Create parking relation
```

So:

```
park(vehicle)
```

matches the direction of causality.

---

# 3. Why unpark takes a Ticket

Now the situation is reversed.

After parking, the authoritative relationship is:

```
Ticket
  |
  +-- vehicle_id
  |
  +-- spot_id
  |
  +-- entry_time
```

The ticket is a **proof of an existing parking transaction**.

Unparking asks:

> "Which parking relationship should I terminate?"

You already know the answer from the ticket.

```
unpark(ticket):
    spot = ticket.spot_id
    release(spot)
    calculate_fee(ticket)
    close(ticket)
```

This is O(1).

---

# 4. Why not unpark(vehicle)?

Imagine:

```python
unpark(vehicle)
```

The system has to ask:

```
Where is this vehicle parked?
```

So you need a lookup:

```
vehicle_id → active_ticket → spot
```

which means either:

* maintain another index
* scan all tickets
* handle duplicate plates, errors, etc.

The vehicle is not the primary identity of the parking event.

---

# 5. The deeper DDD principle: identify the aggregate by the lifecycle event

A useful rule:

> **Commands should enter an aggregate using the identity that uniquely represents the current lifecycle stage.**

Lifecycle:

```
Before parking:

Vehicle
   |
   | park()
   v

ParkingSession (Ticket)
   |
   | unpark()
   v

Completed
```

The "root identity" changes as the state machine evolves.

---

# 6. Another way to see it: create vs destroy

Parking creates a relationship:

```
Vehicle + EmptySpot
          |
          v
    ParkingSession
```

Therefore the input is the thing being introduced:

```
park(vehicle)
```

---

Unparking destroys the relationship:

```
ParkingSession
      |
      v
Vehicle + EmptySpot
```

Therefore the input is the relationship identifier:

```
unpark(ticket)
```

---

# 7. General design pattern

This appears everywhere.

## Hotel

Check-in:

```python
check_in(guest)
```

creates:

```
Reservation / Stay
```

Check-out:

```python
check_out(stay_id)
```

---

## Shopping

Create order:

```python
place_order(cart)
```

Later:

```python
cancel_order(order_id)
```

---

## Banking

Open account:

```python
open_account(customer)
```

Later:

```python
close_account(account_id)
```

---

# Compression table

| Operation type                | Input identity                   | Principle                                               |
| ----------------------------- | -------------------------------- | ------------------------------------------------------- |
| Create relationship           | External entity                  | "What is entering the system?"                          |
| Modify/terminate relationship | Relationship ID / transaction ID | "Which existing state transition is being manipulated?" |
| Query state                   | Most selective identifier        | "What lets us locate state efficiently?"                |

---

So in DDD terms, **`park(vehicle)` is a command that creates a new aggregate instance (a Parking Session/Ticket), while `unpark(ticket)` is a command against an existing aggregate instance.**

The deepest principle is:

> **Commands should be expressed in terms of the entity that owns the information necessary to make the state transition valid.**

For parking, that is the **vehicle's constraints**. For unparking, that is the **ticket's identity and history**.


What you are asking for is essentially the **design derivation process**: *how do I discover the right command boundary and identity?* This is one of the central skills in DDD and low-level design.

The chain of thought is not "what objects exist?" but rather **what state transition am I trying to control?**

---

## Step 1: Ask: What is the state machine?

Do not start with classes.

Start with:

```
Empty Spot + Vehicle
        |
        | park()
        v
Occupied Spot + Parking Session
        |
        | unpark()
        v
Empty Spot + Vehicle exits
```

Now you see there is a new thing created:

```
Parking Session (Ticket)
```

This is the thing whose lifecycle matters.

---

## Step 2: Ask: What information is missing before the transition?

For `park`:

Current state:

```
ParkingLot
  spots: empty
```

Input:

```
Vehicle(type=Car)
```

Question:

> What do I need to decide whether this transition is valid?

You need:

* vehicle type
* vehicle identity (for tracking)

You **do not have a ticket yet**.

So the command must be:

```python
park(vehicle)
```

---

## Step 3: Ask: What identity is created by the transition?

After parking:

```
Ticket {
    ticket_id
    vehicle_id
    spot_id
    entry_time
}
```

The parking operation creates a **new relationship**:

```
Vehicle <---- Ticket ----> Spot
```

The ticket becomes the canonical identity of this parking session.

---

## Step 4: Ask: What does the reverse transition need to reference?

For `unpark`:

The intent is:

> "End a specific parking session."

Now ask:

**What uniquely identifies a parking session?**

Candidates:

### Vehicle

```
vehicle_id → active ticket → spot
```

Requires an additional lookup.

Also has ambiguity:

```
Vehicle A parked Monday
Vehicle A parked Tuesday
```

Which parking event?

---

### Spot

```
unpark(spot_id)
```

Question:

> Who is authorized to release this spot?

Anyone who knows the spot number can do it.

Bad domain boundary.

---

### Ticket

```
ticket_id → spot, vehicle, entry time
```

The ticket directly points to the exact relationship.

Therefore:

```python
unpark(ticket)
```

is the natural command.

---

## Step 5: Ask: What is the aggregate's current identity?

This is a very DDD way to think.

The "identity of the thing under control" changes over time.

Before parking:

```
Vehicle
```

After parking:

```
ParkingSession
```

So:

```
park(vehicle)
       |
       v
ParkingSession(ticket)
       |
unpark(ticket)
```

---

## The general design algorithm

Whenever designing an API, run this checklist:

### 1. What state transition is occurring?

```
Before ---- command ----> After
```

---

### 2. What entity is being created or destroyed?

```
A + B → Relationship
```

or

```
Relationship → A + B
```

---

### 3. What information is necessary to validate the transition?

Examples:

* vehicle type?
* account balance?
* reservation date?

---

### 4. After the transition, what becomes the authoritative identity?

Examples:

```
Order ID
Ticket ID
Reservation ID
Session ID
```

---

### 5. For future commands, use the identity of the current lifecycle stage.

---

## Compression rule

A very powerful heuristic:

| Operation                        | Usually takes                         |
| -------------------------------- | ------------------------------------- |
| Create a relationship            | The participating entities            |
| Change an existing relationship  | The relationship's ID                 |
| Destroy an existing relationship | The relationship's ID                 |
| Query                            | The smallest authoritative identifier |

Examples:

```
placeOrder(cart)         → Order
cancelOrder(orderId)

bookFlight(customer)     → Reservation
cancelFlight(reservationId)

login(credentials)       → Session
logout(sessionId)

park(vehicle)            → Ticket
unpark(ticket)
```

---

The biggest mindset shift is:

**Don't design around nouns ("I have Vehicle and Spot, so methods should take Vehicle"). Design around transitions in a state machine. The parameter is the information needed to move from one valid state to another, and over time the authoritative identity often changes.**

This is the same mental model used in DDD aggregates, event sourcing, finite state machines, and even control theory (commands acting on a system state).


## Interview Tips:

This pattern has several formal interpretations depending on the field. The most useful one for software design is to think of it as an **associative entity (relationship entity)** with its own **identity and lifecycle**.

Let's build the vocabulary.

---

# 1. Database / ER Modeling: Associative Entity

In an ER diagram:

Before parking:

```
Vehicle -------- Spot
```

The relationship "is parked in" is not just a boolean relation because it has attributes:

```
Vehicle
    |
    |
ParkingSession (Ticket)
    |
    |
ParkingSpot
```

Formally:

```
Vehicle ---< ParkingSession >--- Spot
```

The middle object is an **associative entity** (also called a **junction entity**, **link entity**, or **bridge entity**).

It represents the relationship itself as a first-class object.

### Why does it exist?

Because the relation has its own data:

```
ParkingSession {
    id
    vehicle_id
    spot_id
    entry_time
    exit_time
    fee
}
```

A plain edge:

```
Vehicle → Spot
```

cannot naturally hold all this state.

---

# 2. Domain-Driven Design: Aggregate / Domain Entity

In DDD terms:

```
Vehicle        ParkingSpot
    \              /
     \            /
      ParkingSession
```

The parking session is a **domain entity** with:

* identity (`ticket_id`)
* lifecycle
* invariants

Examples:

```
entry_time < exit_time
fee must be paid before close
only one active session per spot
```

So after creation, the command target changes:

```
park(vehicle)
        |
        v
ParkingSession(ticket)
        |
        v
unpark(ticket)
```

---

# 3. Relational Algebra: A Relation with Attributes

Mathematically:

Initially you might think:

[
ParkedIn \subseteq Vehicle \times Spot
]

a simple binary relation.

But reality is:

[
ParkingSession \subseteq Vehicle \times Spot \times Time \times Money \times Status
]

It becomes an **n-ary relation**.

The ticket is the identity of one tuple in this relation.

---

# 4. Category Theory: A Morphism Object / Span

Very abstractly:

```
Vehicle  <- ParkingSession -> Spot
```

is a **span**.

A span is an object that relates two objects through two morphisms.

```
          ParkingSession
          /             \
         /               \
   Vehicle             Spot
```

This appears in databases, graph transformations, and applied category theory.

---

# 5. Graph Theory: Edge Promotion (Reification)

A graph edge:

```
Vehicle ----- Spot
```

is turned into a node:

```
Vehicle --- ParkingSession --- Spot
```

This process is called:

* **edge reification**
* **relationship reification**

because the relationship becomes a thing.

---

# The core design pattern

The general pattern is:

```
A + B
 |
 | create relation
 v
Relationship(A, B)
```

where the relationship has:

* its own identity
* state over time
* business rules

---

# Common software examples

| Domain         | Relationship entity   | Before              | After             |
| -------------- | --------------------- | ------------------- | ----------------- |
| Parking        | ParkingSession/Ticket | Vehicle + Spot      | Ticket            |
| Hotel          | Stay / Reservation    | Guest + Room        | Reservation ID    |
| Airline        | Booking               | Passenger + Flight  | Booking ID        |
| Shopping       | Order                 | Customer + Products | Order ID          |
| Banking        | Account               | Customer + Bank     | Account ID        |
| Employment     | EmploymentContract    | Person + Company    | Contract ID       |
| Education      | Enrollment            | Student + Course    | Enrollment ID     |
| Social media   | Friendship            | User + User         | Friendship object |
| Git            | Pull Request          | Branch + Repository | PR ID             |
| Messaging      | Conversation          | Users               | Conversation ID   |
| Authentication | Session               | User + Device       | Session ID        |
| Ride sharing   | Trip                  | Rider + Driver      | Trip ID           |
| Delivery       | Shipment              | Order + Carrier     | Tracking ID       |

---

# How to spot this pattern in design interviews

Ask these questions:

### 1. Is the relationship long-lived?

```
Customer buys Product
```

No → maybe just an event.

```
Customer places Order
```

Yes → make Order an entity.

---

### 2. Does the relationship have attributes?

```
Student enrolled in Course

grade
semester
enrollment_date
```

Yes → create `Enrollment`.

---

### 3. Does the relationship have a lifecycle?

```
Pending
Approved
Completed
Cancelled
```

Yes → create an entity.

---

### 4. Does it need independent commands?

```
cancelOrder(orderId)
payInvoice(invoiceId)
closeTicket(ticketId)
```

Yes → it deserves an identity.

---

# Compression heuristic

```
A ↔ B
```

Ask:

```
Does A-B have:
    state?
    time?
    money?
    permissions?
    status?
    history?
    independent commands?
```

If yes:

```
A ← RelationshipEntity → B
```

Give it:

```
id
state machine
invariants
commands
```

Examples:

```
Vehicle + Spot      -> ParkingSession
Customer + Products -> Order
User + Device       -> Session
Student + Course    -> Enrollment
```

---

This is actually one of the most important recurring patterns in system design. In database design it is **associative entities**, in DDD it is often a **domain entity/aggregate**, in graph theory **edge reification**, and in category theory a **span**. It is the formal move of turning a **relationship into a first-class stateful object**.


## Attempt 2

### requirements:

1. Park and unpark car tracking only, no fees for now

In [ ]:

from enum import Enum
import uuid

class VehicleType(Enum):
    Bike = "Bike"
    Car = "Car"
    Truck = "Truck"

class ParkingSpot:
    def __init__(self):
        self.occupant = None
        self.type = None

    # [2] Mutator for spot occupancy. Local enforcement point: only assign when the spot
    # is empty and the vehicle type is compatible with this spot.
    def assign(self, vehicle):
        if self.occupant is None and vehicle.type == self.type:
            self.occupant = vehicle
            return True
        else:
            return False
    
    # check unpark for later.
    def release(self, vehicle):
        if self.occupant == vehicle:
            self.occupant = None
            return True
        else:
            return False
    
    def get_occupant(self):
        return self.occupant

# TODO: design vehicle
class Vehicle:
    def __init__(self, id = None, type = None, ticket = None):
        self.id = id
        self.type = type
        self.ticket = ticket
        

# TODO: design Parking Ticket.

class ParkingTicket:
    def __init__(self, spot_id = None, vehicle_id = None, entry_time = None, floor_id = None):
        self.id = str(uuid.uuid4())
        self.spot_id = spot_id
        self.vehicle_id = vehicle_id
        self.entry_time = entry_time
        self.floor_id = floor_id


class ParkingFloor:
    def __init__(self,size):
        self.parking_grid = {spot_id: ParkingSpot(type = 'truck' if spot_id % 2 == 0 else 'car') for spot_id in range(size//2)}
        self.size = size
        self.filled = 0

    def full(self):
        return self.filled == self.size

    def park(self, vehicle):
        if not self.full():
            parking_spot = None
            for spot_id, spot in self.parking_grid.items():
                if spot.assign(vehicle):
                    parking_spot = spot_id
                    break
            if parking_spot is not None:
                self.filled += 1
                ticket = ParkingTicket(spot_id = parking_spot, vehicle_id = vehicle.id, entry_time = int(time.time()), id=uuid.uuid4())
                self.parking_grid[parking_spot].occupant.ticket = ticket
                return ticket

    def unpark(self, ticket):
        parking_spot = self.parking_grid.get(ticket.spot_id)
        if parking_spot and parking_spot.release(parking_spot.get_occupant()):
            self.filled -= 1
            del parking_spot.occupant.ticket
            return True
        return False
    

class ParkingLot:
    def __init__(self, parking_floors = 10, parking_floor_size = 10):
        self.parking_floors = [ParkingFloor(size = parking_floor_size)]
        self.tickets = {}
    def park(self, vehicle):
        parked = False
        for parking_floor in self.parking_floors:
            ticket = parking_floor.park(vehicle)
            if ticket:
                parked = True
                self.tickets[ticket.id] = ticket
                break
        return parked

    def unpark(self,ticket):
        if ticket.id in self.tickets and :
            unparked = False
            for parking_floor in self.parking_floors:
                if parking_floor.unpark(self.tickets[ticket.id]):
                    unparked = True
                    del self.tickets[ticket.id]
                    break
            return unparked
        return False


### Critique After Attempt 2 (latest)

1. Findings

- High: Requirements are narrowed to "park/unpark car tracking only, no fees," but the implementation still declares Bike/Car/Truck without a clear scope contract. Lock scope explicitly (single type vs multi-type) before modeling entities.
- High: Invariants are missing as always-true statements. Right now correctness is implicit in methods, so there is no explicit guarantee for core rules like "one active ticket per parked vehicle" and "one occupied vehicle per spot." 
- High: Ownership is still unstable. `ParkingFloor` performs lifecycle mutation (`park/unpark`) while `ParkingLot` also owns global active-ticket state. This split makes invariant enforcement ambiguous.
- High: The state machine is not explicit. Legal/illegal transitions (e.g., unpark with stale/forged ticket, double unpark, park when full) are not modeled first, so behavior is ad hoc in method branches.
- High: Core model has execution-breaking mismatches: `ParkingSpot.__init__` takes no `type` but is called with `type=...`; `time` is used but not imported; `ParkingTicket` constructor is called with unsupported `id` argument; `ParkingFloor(size)` creates only `size//2` spots but `full()` compares against `size`.
- Medium: Ticket identity flow is inconsistent. `ParkingLot.unpark` accepts a ticket object, but true authority should be `ticket_id -> active session` lookup; current API makes forged object usage easier and weakens boundary checks.
- Medium: `Vehicle.ticket` as mutable attached state is not clearly owned. During unpark, ticket cleanup mutates occupant internals directly (`del ...ticket`), which couples spot release with vehicle mutation authority.
- Medium: DS/operations are only partially aligned. Linear scan is fine for now, but "nearest available spot" is a requirement and is not represented in structure or ordering policy.
- Medium: Happy/failure traces are missing for this latest attempt revision, so ownership and transition gaps are not validated end-to-end.

2. Revision order

1. Rewrite section 1 with explicit scope contract for this iteration (single floor? multi-floor? which vehicle types? no fee yet).
2. Add 3 to 5 invariants as always-true statements, each with an owner that enforces it.
3. Write the spot/session state machine with legal and illegal transitions before class edits.
4. Rewrite responsibilities in `Rule -> Owner -> Mutator -> Enforcement` form, especially ticket lifecycle and unpark authority.
5. Refactor API boundary to `unpark(ticket_id)` and centralize active-session validation in `ParkingLot`.
6. Re-run one happy path and one failure path trace, then patch code to match those traces.


## 5 questions to Challenge my understanding


  1. Which single object is the mutation authority for transitioning a spot from empty to
     occupied, and what invariant would fail first if another object also mutates it?

   ans. The mutation authority should be owned by ParkingLot itself, but state tracking should be done by ParkingSpot. The Invariant of occupation, track of local and hence global ParkingLot would be violated

  2. If unpark receives a forged or stale ticket id, what exact enforcement point rejects
     it, and what state remains unchanged after rejection?
     ans. the parking lot ticket hashmap should enforce valid ticket ids. The occupancy should remain unchanged after rejection.

  3. What are your legal and illegal transitions for one parking session from creation to
     close, including the double-unpark case?
     ans. Legal: if ticket is valid then unpark, if there is no ticket or another ticket shares same parking spot or vehicle id then illegal.

  4. For your current DS choices, what is the time complexity of park and unpark, and which
     requirement would force you to change the structure first?
      ans. Park is O(ParkingFloor.size \times ParkingLot.parking_floors)
      If the parking is ordered within the floor that the vehicle prefers to park some ordered spot, then I'd have to change the structure from dictionary within the grid to list or heap + hashmap where hashmap stores valid while, heap stores remaining spaces in priority order.

  5. If the requirement changes to “support nearest-spot allocation across floors,” which
     class boundaries stay stable and which ownership rules must move?
     ans. I don't get what class boundaries mean, however, I'm thinking ParkingLot handles the allocation logic including parking ticket creation. while parkingfloor

### Critique of your 5-question responses and guidance before Attempt 3

1. Findings

- High: You are still mixing **state owner** with **mutation authority**. Saying `ParkingLot` owns the mutation while `ParkingSpot` tracks the state is only safe if you state the rule precisely: `ParkingSpot` owns occupancy state, while `ParkingLot` orchestrates transitions and validates system-level invariants before calling into the spot. Without that split, ownership is still ambiguous.
- High: Your answer to the stale/forged ticket question is directionally correct but incomplete. The rejection point is not just "the hashmap"; it is the `ParkingLot.unpark(ticket_id)` validation boundary that checks `ticket_id -> active session` before any floor or spot mutation. You must name what does **not** change: ticket map, spot occupancy, and any vehicle-session association.
- High: Your lifecycle answer is still under-modeled. `valid ticket then unpark` is not a state machine. You need explicit session states and transitions, for example `NONE -> ACTIVE -> CLOSED`, plus illegal operations like `CLOSED -> CLOSED` via double unpark and `NONE -> CLOSED` via forged ticket.
- Medium: Your DS answer shows decent instincts, but you skipped `unpark` complexity and the current authoritative lookup shape. With `ticket_id -> active ticket`, unpark should target O(1) session lookup plus direct floor/spot resolution from the ticket. If you need to scan floors on unpark, your identity model is still weak.
- Medium: You identified `ParkingLot` as the right place for allocation logic, but you do not yet have a clean statement of what stays stable under the nearest-spot change. The stable boundary is roughly: `ParkingLot` remains the orchestration root, `ParkingFloor` remains local spot inventory, `ParkingSpot` remains occupancy state holder. The moving part is allocation policy and any data structure supporting ordered selection.
- Medium: You are still answering with implementation fragments before locking the drill artifacts. The earlier critique asked for requirements, invariants, state machine, ownership table, and traces first. Attempt 3 should not jump back into code until those are explicit.

2. More guidance

1. Rewrite ownership with one sentence per mutable state: `spot occupancy is owned by ParkingSpot`; `active parking sessions are owned by ParkingLot`; `floor inventory is owned by ParkingFloor`.
   Formalization: this split follows the rule that each mutable part of the abstract state should have one primary state owner in `X`, while orchestration may live elsewhere. `ParkingSpot` owns occupancy because occupancy is a local spot-state predicate (`empty` or `occupied(vehicle)`), and keeping that field on the spot makes the occupancy invariant checkable at the state carrier. `ParkingLot` owns active parking sessions because the session index (`ticket_id -> active session`) is a system-level representation that crosses floors and is the guard point for legal `unpark(ticket_id)` transitions. `ParkingFloor` owns floor inventory because membership of spots to a floor is floor-local structure, and nearest-spot or compatibility scans depend on that local collection boundary. In formal terms, the ownership map should minimize diffuse mutation authority: local state stays with the smallest stable carrier, while cross-cutting indexes and transition guards stay at the aggregate root.
2. Write invariants as always-true claims, not workflow steps. Good examples here are: `an occupied spot has exactly one active ticket`, `an active ticket refers to exactly one occupied spot`, `a closed ticket cannot be used to unpark again`.
3. Write the parking-session state machine separately from the spot state machine. You are currently collapsing them into one vague notion of "valid ticket". For the current simplified requirements, you do not need a large lifecycle, but you still need at least the conceptual distinction between `no session`, `active session`, and `closed/nonexistent for active use`. Otherwise you cannot state clearly what a stale ticket means or why double-unpark is illegal. If you want a minimal model, it is acceptable to represent this as `ticket exists in active_sessions` vs `ticket does not exist in active_sessions`, which is effectively a 2-state session model. That means yes: for this round, you can avoid storing historical closed tickets if fees, receipts, audits, and reporting are out of scope. But the design must still model the transition from active to inactive explicitly. In many real business systems, tickets are retained after close for billing disputes, audit logs, receipts, analytics, and lost-ticket handling, which is why richer ticket states are common even if your current exercise does not require them.
4. Make `unpark(ticket_id)` the public boundary for Attempt 3. Let `ParkingLot` validate the active session, then delegate the physical release to the floor/spot level.
5. Run one explicit happy path and one failure path in text before coding: `park(car)` success, and `unpark(stale_ticket_id)` rejection. This will force you to say whether a stale ticket means `closed ticket still stored` or `ticket no longer present in active_sessions`; both are valid models, but you need to choose one deliberately.
6. Treat requirement changes as a boundary test: ask which existing owner keeps its job and which policy or DS must change. That is what `class boundaries stay stable` means in this context.

3. Gap matrix

| Area | What you currently understand | Current gap | What to fix in Attempt 3 |
| --- | --- | --- | --- |
| Requirements | You know the scope must be narrowed per iteration. | Your scope is still half-specified and drifts between simplified and full problem requirements. | State exact scope in 3 to 5 bullets before design. |
| Invariants | You recognize occupancy consistency matters. | You describe consequences loosely instead of writing enforceable always-true statements. | Write 3 to 5 invariants with a named owner for each. |
| State machine | You know validity of ticket matters. | You have not enumerated states, legal transitions, or illegal transitions. | Write session and spot transitions explicitly. |
| Ownership | You sense `ParkingLot` should orchestrate more. | You still blur owner, mutator, and enforcer. | Fill `Rule -> Owner -> Mutator -> Enforcement` for park and unpark. |
| Identity / API boundary | You understand ticket lookup is important. | You still phrase unpark around a vague ticket object or validity notion. | Use `ticket_id` as the external command input and map it to an active session. |
| Data structures | You see ordered allocation may require better DS. | You did not connect DS choice to current operations, especially unpark. | State `park` and `unpark` target complexity and choose DS to support them. |
| Extensibility | You correctly place nearest-spot logic near lot-level orchestration. | You do not yet separate stable boundaries from changing policy. | Keep owners stable; move only allocation strategy and supporting indexes. |
| Trace discipline | You can reason locally about method behavior. | You are still not validating the whole design through one happy and one failure path. | Write both traces before touching code. |




## TODOs before Attempt 3 code

1. Write a 3 to 5 bullet scope section.
   Why: your requirements are still drifting between a simplified exercise and the full parking-lot problem, which makes the rest of the design unstable.
   1. Park(vehicle), unpark(ticket) is handled by ParkingLot. No fee calculation for now, but entry and exit time is ok, (perhaps this is shown on the parking ticket upon exit)
   2. ParkingLot is the main entry point for all operations
   3. unpark should return a fee if applicable
2. Write 3 to 5 invariants with an explicit owner for each.
   Why: the critique shows you understand consequences of bad state, but not yet the always-true rules that the design must preserve.
   1. ParkingLot.tickets only tracks valid tickets
   2. (ParkingFloor[i][ParkingSpot]) always tracks occupancy to the exact time in coordination with parkingTicket of the vehicle within 1 to 1 mapping  of active tickets.
   
3. Write a minimal session state model and a spot state model.
   Why: you still describe validity informally, so stale ticket handling and double-unpark rejection are not grounded in explicit states.
   ParkingLot.tickets will store active and validated tickets for audit history, park creates ticket which is active, and unpark validates the ticket and marks it as inactive.

4. Fill a `Rule -> Owner -> Mutator -> Enforcement` table for `park` and `unpark`.
   Why: your biggest conceptual gap is still mixing state ownership, mutation authority, and orchestration.
   ans. I don't get this question. but the ticketstatus within the self.tickets active inactive and leaving unpark is inactive if precondition was active is enforced in ParkingLot. And I also made parkinglot own the ParkingTicket creation instead of Parkingfloor 

5. Choose the external unpark API as `unpark(ticket_id)` and state what `active_sessions` stores.
   Why: your identity boundary is still underspecified, and that boundary determines both legality checks and unpark complexity.

   ans. i decided to use self.tickets which stores full audit history of tickets, activity is by ticket.status. legality check is O(1) by hashmap in this case

6. State target time complexity for `park` and `unpark`, then justify the DS.
   Why: your DS answer had the right instinct but was not yet tied to the actual operations and current representation.

   ans. park is O(N x M) where N is the number of floors and M is the number of spots per floor, unpark is O(1) by hashmap

7. Write one happy path and one failure path in plain text.
   Why: the critique keeps finding missing end-to-end validation, and traces are the fastest way to expose ownership mistakes before coding.

   ans. I'm not sure if the vehicle should store the ticket or not in the realworld, of course the ticket is stored by global parkinglot and stores the vehicle for validation invalidation, since the controller is from parkinglot. for simplification since we're doing parkinglot instead of vehicle control side implemenation, I'm making vehicle not store the ticket

8. Only after the above, revise the code cell to match the written model.
   Why: your current pattern is still code-first, while the earlier critique and this one both show the model is not stable enough yet.

## Attempt 3 (more guidance)

3. explicit scope contract: ParkingLot manages multiple floors, and for the moment cars and trucks of the specific configuration half half. later what should it be?
a. ParkingSpot owns spot occupancy, ParkingLot owns active parking sessions, floor inventory owned by parking floor. 

4. occupied spot <-> active ticket 1 to 1, closed ticket can't be used to unpark again




In [ ]:

from enum import Enum
import uuid


class VehicleType(Enum):
    Bike = "Bike"
    Car = "Car"
    Truck = "Truck"

# TODO: design vehicle
class Vehicle:
    def __init__(self, id = None, type: VehicleType = None, ticket = None):
        self.id = id
        self.type = type


class ParkingSpot:
    def __init__(self, type: VehicleType):
        self.occupant = None
        self.type = type

    # [2] Mutator for spot occupancy. Local enforcement point: only assign when the spot
    # is empty and the vehicle type is compatible with this spot.
    def assign(self, vehicle):
        if self.occupant is None and vehicle.type == self.type:
            self.occupant = vehicle
            return True
        else:
            return False
    
    # [5] Mutator for spot release. Local enforcement point: only release when the
    # occupant identity matches the expected vehicle_id.
    def release(self, vehicle_id) -> Vehicle:
        if self.occupant and self.occupant.id == vehicle_id:
            ret = self.occupant
            self.occupant = None
            return ret
        else:
            return None
    
    def get_occupant(self):
        return self.occupant


class TicketStatus(Enum):
    Active = "Active"
    Inactive = "Inactive"    


class ParkingTicket:
    def __init__(self, spot_id = None, vehicle_id = None, entry_time = None, floor: int = None):
        self.id = str(uuid.uuid4())
        self.spot_id = spot_id
        self.vehicle_id = vehicle_id
        self.entry_time = entry_time
        self.exit_time = entry_time
        self.floor = floor
        self.status = TicketStatus.Active


class ParkingFloor:
    def __init__(self,size):

        self.parking_grid = {spot_id: ParkingSpot(type = 'truck' if spot_id % 2 == 0 else 'car') for spot_id in range(size//2)}
        self.size = size
        self.filled = 0

    def full(self):
        return self.filled == self.size

    # [3] Mutator for floor-local occupancy count on successful park.
    # Enforcement point: only increment filled after a real spot assignment succeeds.
    def park(self, vehicle):
        if not self.full():
            parking_spot = None
            for spot_id, spot in self.parking_grid.items():
                if spot.assign(vehicle):
                    parking_spot = spot_id
                    break
            if parking_spot is not None:
                self.filled += 1
                return spot_id

    # [3] Mutator for floor-local occupancy count on successful unpark.
    # Enforcement point: only decrement filled after the matching spot release succeeds.
    def unpark(self, ticket) -> Vehicle:
        parking_spot = self.parking_grid.get(ticket.spot_id)
        if parking_spot:
            vehicle = parking_spot.release(ticket.vehicle_id)
            if vehicle:
                self.filled -= 1
                return vehicle
        return None
    

class ParkingLot:
    def __init__(self, parking_floors = 10, parking_floor_size = 10):
        self.parking_floors = [ParkingFloor(size = parking_floor_size) for _ in range(parking_floors)]
        self.tickets = {} # tracks all active and inactive tickets
        self.time = 0
        self.total_earnings = 0

    # [1] Aggregate command boundary for creating a new parking session.
    # Enforcement point: only create and insert a ticket after floor allocation succeeds.
    def park(self, vehicle):
        ticket = None
        for idx, parking_floor in enumerate(self.parking_floors):
            spot_id = parking_floor.park(vehicle)
            if spot_id:
                ticket = ParkingTicket(spot_id=spot_id, vehicle_id=vehicle.id, entry_time=self.time, floor=idx)
                self.tickets[ticket.id] = ticket
                break
        return ticket

    def calculate_fees(self, ticket):
        # direct 1 per hour cost
        return ticket.exit_time - ticket.entry_time 

    def time_increment(self):
        self.time += 1 
        
    # [4] Aggregate command boundary for rejecting stale/inactive tickets and closing
    # an active session.
    # [6] Aggregate enforcement point for preserving the 1-to-1 mapping between active
    # tickets and occupied spots across floors.
    def unpark(self,ticket):
        if ticket.floor < len(self.parking_floors) and ticket.id in self.tickets and self.tickets[ticket.id].status == TicketStatus.Active:
            vehicle = self.parking_floors[ticket.floor].unpark(self.tickets[ticket.id])
            if vehicle:
                ticket.fees = self.calculate_fees(vehicle.ticket)
                self.total_earnings += ticket.fees
                ticket.status = TicketStatus.Inactive
                return ticket.fees
        return None


### Critique After Attempt 3

1. Findings

- High: your current code still violates the narrowed model in execution-critical ways. `ParkingFloor.parking_grid` builds spot types as raw strings (`'truck'`, `'car'`) while `Vehicle.type` is a `VehicleType` enum, so `ParkingSpot.assign()` will reject every vehicle.
- High: `ParkingLot.park()` treats `spot_id` as truthy, so a valid spot at index `0` is treated as failure. This is a state transition bug, not a syntax issue.
- High: you intentionally stopped storing `vehicle.ticket`, but `ParkingLot.unpark()` still writes through `vehicle.ticket.exit_time`. That means your ownership direction improved, but the code path still depends on the old model and will fail.
- High: your floor capacity model is still inconsistent. `ParkingFloor` creates `size//2` spots but `full()` compares `filled == size`, so reachability and capacity invariants cannot hold.
- High: the external unpark boundary is still wrong for the design you said you want. The code uses `unpark(ticket)` instead of `unpark(ticket_id)`, which keeps identity and legality checks weaker than your own Attempt 3 notes.
- Medium: progression is real and positive. Moving ticket creation to `ParkingLot`, adding `TicketStatus`, and making the ticket carry `floor` are all better ownership signals than Attempt 2. Those are structural improvements, not cosmetic ones.
- Medium: your requirement bullets are still unstable. You say "no fee calculation for now" and also "unpark should return a fee if applicable". That ambiguity leaks directly into `calculate_fees`, `exit_time`, and `total_earnings`.
- Medium: your invariants are closer, but they are still partly workflow descriptions. `ParkingLot.tickets only tracks valid tickets` conflicts with your later statement that the map stores audit history with active and inactive tickets. Pick one representation and state it precisely.
- Medium: your inline comment intuition is mixed. `# shifted ownership of ticket creation to parkinglot too. but why?` is asking the right design question; the answer is that ticket issuance is a cross-floor session creation step guarded by lot-level invariants. In contrast, `#give the vehicle` on `release()` is not a design insight; it hides the more important question, which is whether `release` should consume `vehicle_id`, `ticket`, or no external identity at all.
- Medium: you still have not written the happy path and failure path as traces before coding, and that is why old-model and new-model assumptions are now mixed inside the same methods.

2. Progression critique

- Stronger than Attempt 2: you moved session creation upward to `ParkingLot`, introduced explicit active/inactive ticket status, and started separating session ownership from spot occupancy.
- Still not stable: you changed some code to the new ownership model, but not all dependent paths. That means the design is improving faster than the implementation discipline.
- Main pattern in your progression: your intuition about ownership is getting better, but you still convert that intuition into code before the full transition model is written down.

3. In-code comment intuition check

| Location | Intuition quality | Critique |
| --- | --- | --- |
| `# TODO: design vehicle` | Medium | Reasonable placeholder, but the real question is whether `Vehicle` is only identity plus type for this problem or whether it also owns parking-session state. Your later code suggests it should not own the ticket. |
| `# contracts: true if operation worked as expected, else false.` | Medium | The intuition about method contracts is fine, but it is underspecified. The design needs preconditions and invariant preservation, not just success booleans. |
| `# check unpark for later. #give the vehicle` | Low | This comment does not identify the important design choice. The core question is who authorizes release and what identity the release consumes. |
| `# shifted ownership of ticket creation to parkinglot too. but why?` | High | Good intuition. Ticket issuance belongs with the aggregate root because it creates the cross-floor active session index and enforces lot-level uniqueness and legality. |
| `self.tickets = {} #tracks all active and inactive tickets tickets` | Medium | The comment identifies the intended representation, but it conflicts with your earlier invariant wording about only valid tickets. Tighten the invariant to match the representation. |
| `# direct 1 per hour cost` | Low | This is implementation-first while requirements are still unstable. Until fees are clearly in scope, this comment introduces accidental product behavior. |

4. Gap matrix

| Area | Progress since Attempt 2 | Remaining gap | What to change next |
| --- | --- | --- | --- |
| Requirements | You are trying to narrow scope before coding. | Fee support is still contradictory. | Decide now: fees in scope or out of scope for Attempt 3. |
| Invariants | You started expressing 1-to-1 ticket/spot thinking. | Invariants still conflict with the chosen ticket-history representation. | Rewrite invariants against the actual data model. |
| State machine | You introduced `TicketStatus`. | Spot lifecycle and stale-ticket handling are still not written as explicit transitions. | Write `spot` and `ticket/session` state machines before more code changes. |
| Ownership | You moved ticket creation to `ParkingLot`. | Old ownership assumptions still survive in `unpark()`. | Remove `vehicle.ticket` dependence and centralize validation in lot-level logic. |
| API boundary | You understand ticket identity matters more. | Public API still accepts a ticket object. | Change to `unpark(ticket_id)` and keep ticket lookup internal. |
| Data representation | You introduced `status` and `floor` on tickets. | Representation still has type mismatch and capacity mismatch bugs. | Make `VehicleType` consistent end-to-end and fix floor capacity math. |
| Trace discipline | You are asking better design questions in comments. | You still have no explicit happy/failure traces. | Write both traces and then align code line-by-line. |

5. Revision order

1. Resolve the scope contradiction around fees. If fees are out, remove `calculate_fees`, `total_earnings`, and `exit_time` handling for now.
2. Rewrite invariants to match one chosen ticket model: either `tickets stores active only` or `tickets stores all sessions and status distinguishes active vs inactive`.
3. Write the minimal state machines explicitly: `spot: EMPTY -> OCCUPIED -> EMPTY`; `session: ABSENT -> ACTIVE -> INACTIVE` or `ticket_id not in map` if you choose deletion instead of retention.
4. Change the external command to `unpark(ticket_id)` and make `ParkingLot` do `ticket_id -> session -> floor/spot` resolution.
5. Fix the representation mismatches before any further design judgment: enum-vs-string spot types, `spot_id == 0` handling, floor capacity math, and `vehicle.ticket` removal.
6. Add one happy path and one failure path trace in text, then check each line of code against the trace.

6. Challenge questions

1. If `tickets` stores both active and inactive sessions, what exact invariant distinguishes a reusable stale id from an auditable closed session?
2. Where is the single enforcement point that prevents a ticket from being closed twice, and what state is unchanged when that enforcement rejects the second close?
3. If you remove `vehicle.ticket` entirely, which object now carries enough information to compute exit behavior and why is that ownership cleaner?
4. Under your current representation, what must be true about `ticket.floor` and `ticket.spot_id` for `unpark(ticket_id)` to avoid scanning unrelated floors or spots?
5. If fees are out of scope today but added tomorrow, which part of `delta` changes and which invariants should stay unchanged?

7. Optional deeper model

Formalization of your current design direction:

- `Sigma` (abstract state): spot occupancy across floors, active/inactive parking-session status, current lot time, and optionally earnings if fees are in scope.
- `X` (representation): `parking_floors`, each floor's `parking_grid`, and `tickets` as the session index. If you keep audit history, `tickets` represents all sessions plus status; if not, it represents active sessions only.
- `I` (invariants): `each occupied spot corresponds to exactly one active session`; `each active session refers to exactly one occupied spot`; `an inactive session cannot trigger another unpark`; `filled` equals the number of occupied spots on a floor`.
- `delta` (transitions): `park(vehicle)` should create a legal `EMPTY -> OCCUPIED` spot transition and `ABSENT -> ACTIVE` session transition together; `unpark(ticket_id)` should create `OCCUPIED -> EMPTY` and `ACTIVE -> INACTIVE` together or reject without mutation.
- `Phi` (guards): spot compatibility, capacity, ticket existence, ticket active status, and floor/spot reference validity.
- `A` (mutation authority sketch): `ParkingSpot` owns occupancy field mutation; `ParkingFloor` owns local spot inventory lookup; `ParkingLot` owns session creation/closure authority and legality checks for external commands.

The key formal mistake still present in code is transition non-preservation: the new ownership model in your notes is not yet preserved by the actual `unpark()` implementation, because the code still depends on `vehicle.ticket` and a ticket-object boundary.


## Couple Notes to self + Requirements

1. Validation within Aggregate is crucial - ensure invariants are maintained
2. ParkingLot tracks all tickets throughout history.
3. Scope will calculate fees in this basic manner by the aggregate
4. tickets stored in parkinglot will be both active and inactive, where the status of the tickets are tracked by the ticket itself, which staleness is tracked by parkinglot.ticket.status

In [ ]:
from enum import Enum
import uuid


class VehicleType(Enum):
    Bike = "Bike"
    Car = "Car"
    Truck = "Truck"

class Vehicle:
    def __init__(self, id = None, type: VehicleType = None, ticket = None):
        self.id = id
        self.type = type


class ParkingSpot:
    def __init__(self, type: VehicleType):
        self.occupant = None
        self.type = type

    # [2] Mutator for spot occupancy. Local enforcement point: only assign when the spot
    # is empty and the vehicle type is compatible with this spot.
    def assign(self, vehicle):
        if self.occupant is None and vehicle.type == self.type:
            self.occupant = vehicle
            return True
        else:
            return False
    
    # [5] Mutator for spot release. Local enforcement point: only release when the
    # occupant identity matches the expected vehicle_id.
    def release(self, vehicle_id) -> Vehicle:
        if self.occupant and self.occupant.id == vehicle_id:
            ret = self.occupant
            self.occupant = None
            return ret
        else:
            return None
    
    def get_occupant(self):
        return self.occupant


class TicketStatus(Enum):
    Active = "Active"
    Inactive = "Inactive"    


class ParkingTicket:
    def __init__(self, spot_id = None, vehicle_id = None, entry_time = None, floor: int = None):
        self.id = str(uuid.uuid4())
        self.spot_id = spot_id
        self.vehicle_id = vehicle_id
        self.entry_time = entry_time
        self.exit_time = entry_time
        self.floor = floor
        self.status = TicketStatus.Active


class ParkingFloor:
    def __init__(self,size):

        self.parking_grid = {spot_id: ParkingSpot(type = VehicleType.Bike if spot_id % 2 == 0 else VehicleType.Car) for spot_id in range(size)}
        self.size = size
        self.filled = 0

    def full(self):
        return self.filled == self.size

    # [3] Mutator for floor-local occupancy count on successful park.
    # Enforcement point: only increment filled after a real spot assignment succeeds.
    def park(self, vehicle):
        if not self.full():
            parking_spot = None
            for spot_id, spot in self.parking_grid.items():
                if spot.assign(vehicle):
                    parking_spot = spot_id
                    break
            if parking_spot is not None:
                self.filled += 1
                return spot_id

    # [3] Mutator for floor-local occupancy count on successful unpark.
    # Enforcement point: only decrement filled after the matching spot release succeeds.
    def unpark(self, ticket) -> Vehicle:
        parking_spot = self.parking_grid.get(ticket.spot_id)
        if parking_spot:
            vehicle = parking_spot.release(ticket.vehicle_id)
            if vehicle:
                self.filled -= 1
                return vehicle
        return None
    

class ParkingLot:
    def __init__(self, parking_floors = 10, parking_floor_size = 10):
        self.parking_floors = [ParkingFloor(size = parking_floor_size) for _ in range(parking_floors)]
        self.tickets = {} # tracks all active and inactive tickets
        self.time = 0
        self.total_earnings = 0

    # [1] Aggregate command boundary for creating a new parking session.
    # Enforcement point: only create and insert a ticket after floor allocation succeeds.
    def park(self, vehicle):
        ticket = None
        for idx, parking_floor in enumerate(self.parking_floors):
            spot_id = parking_floor.park(vehicle)
            if spot_id:
                ticket = ParkingTicket(spot_id=spot_id, vehicle_id=vehicle.id, entry_time=self.time, floor=idx)
                self.tickets[ticket.id] = ticket
                break
        return ticket

    def calculate_fees(self, ticket):
        # direct 1 per hour cost
        return ticket.exit_time - ticket.entry_time 

    def time_increment(self):
        self.time += 1 
        
    # [4] Aggregate command boundary for rejecting stale/inactive tickets and closing
    # an active session.
    # [6] Aggregate enforcement point for preserving the 1-to-1 mapping between active
    # tickets and occupied spots across floors.
    def unpark(self,ticket):
        if ticket.floor < len(self.parking_floors) and ticket.id in self.tickets and self.tickets[ticket.id].status == TicketStatus.Active:
            vehicle = self.parking_floors[ticket.floor].unpark(self.tickets[ticket.id])
            if vehicle:
                ticket.exit_time = self.time
                ticket.fees = self.calculate_fees(ticket)
                self.total_earnings += ticket.fees
                ticket.status = TicketStatus.Inactive
                return ticket.fees
        return None


### Final Critique After Couple Notes to self + newest attempt

1. Findings

- High: your notes and code are now more aligned on ticket history, but the execution path still breaks that model. `Vehicle` no longer stores a ticket, yet `calculate_fees(vehicle.ticket)` is still called during `unpark()`. That means the ownership decision is better, but the implementation has not fully absorbed it.
- High: `ParkingLot.park()` still treats `spot_id` as truthy, so spot `0` is lost. This is a pure correctness bug and will hide successful parking on the first spot of any floor.
- High: your current requirements and code still disagree on vehicle/spot compatibility. The notes say cars and trucks; the newest code builds `Bike` and `Car` spots. That is still a requirements instability problem, not just a code bug.
- Medium: the newest notes are stronger than the previous attempt. `ParkingLot tracks all tickets throughout history` and `status tracked by the ticket itself` is now a coherent representation choice. That is a real design improvement.
- Medium: `Validation within Aggregate is crucial` is the right instinct, but you still have not converted it into an explicit owner/enforcement table. The design intuition is ahead of the written model.
- Medium: `Consider using state pattern for vehicle states` is not the right next move for the current problem. Vehicle state is not the unstable part here; session lifecycle and aggregate validation are. This is premature abstraction.
- Medium: fee handling is now more clearly in scope, which is better than before, but the fee model is still underspecified. You need to state whether `exit_time` is set on the ticket during unpark and whether closed tickets retain fee history.
- Medium: your API boundary is still behind your design maturity. You now understand stale tickets via `ticket.status`, but the public boundary still accepts a `ticket` object instead of `ticket_id`, which weakens external validation semantics.
- Medium: the trace gap remains. The latest attempt still has no explicit happy path or failure path written out, so code and model are being reconciled only indirectly.

2. Revision order

1. Lock the scope in one short block.
   Why: your notes say cars and trucks, but the code currently models bikes and cars.
2. Rewrite 3 to 5 invariants against the current chosen representation.
   Why: you have now chosen `tickets stores active and inactive history`, so the invariants should reflect that exact model.
3. Change the external API to `unpark(ticket_id)`.
   Why: this is the cleanest enforcement point for stale-ticket rejection and keeps session lookup authoritative inside `ParkingLot`.
4. Remove the leftover `vehicle.ticket` dependency and compute fees from the resolved ticket object.
   Why: your ownership model now says the session is owned by `ParkingLot`, not by `Vehicle`.
5. Fix the `spot_id == 0` bug and keep enum usage consistent.
   Why: these are current correctness blockers independent of higher-level design quality.
6. Write one happy path and one failure path in text.
   Why: you are close enough now that traces will expose the remaining inconsistencies quickly.
7. Add the ownership table: `Rule -> Owner -> Mutator -> Enforcement`.
   Why: this is still the main missing artifact between your good intuition and a stable LLD answer.

3. Challenge questions

1. If `tickets` stores full history, what exact predicate defines whether a ticket is stale versus merely old but valid for audit?
2. At what single line of authority should `ticket.status` change from `Active` to `Inactive`, and what other state must change atomically with it?
3. If `Vehicle` does not own session state, what is the minimum data `Vehicle` should still carry for this scoped problem?
4. What is the invariant relating `filled` to `parking_grid`, and where is it enforced on both `park` and `unpark`?
5. If you switch the public API to `unpark(ticket_id)`, which checks happen before any floor mutation occurs?

4. Optional progression critique

- Attempt 1 to Attempt 2: major jump in recognizing aggregate-root ownership and ticket identity.
- Attempt 2 to Attempt 3: major jump in representing session lifecycle explicitly with `TicketStatus`.
- Newest notes to newest code: partial consolidation. You are no longer just reacting to critique; you are choosing a representation (`tickets` keeps full history) and carrying it into code. That is good progress.
- Remaining issue in your progression: you are still stabilizing the model in your head faster than you stabilize the external command boundary and the invariants on paper.

5. Optional deeper model

- Closeness rating: about `6/10` overall for a solid scoped interview answer.
- Design understanding: about `7/10`; you now have the right center of gravity around aggregate validation, session lifecycle, and lot-level ownership.
- Drill completeness: about `5/10`; the missing artifacts are still explicit invariants, an ownership table, and written traces.
- Code/model alignment: about `4/10`; there are still old-model leftovers and correctness bugs.

You are no longer far away. The remaining distance is mostly formalization and consistency, not basic discovery.


### Spot 0 bug, invariants, and owner/enforcement table

1. What `spot_id` as truthy means

In Python, `0` is treated as `False` in boolean checks. That matters in your current `ParkingLot.park()` implementation:

```python
spot_id = parking_floor.park(vehicle)
if spot_id:
    ticket = ParkingTicket(...)
```

In your latest exported code this is around `ParkingLot.park()` line 101.

If the first valid parking spot has id `0`, then `parking_floor.park(vehicle)` returns `0`, but `if spot_id:` treats that as false. So the lot behaves as if parking failed, even though the floor already assigned the vehicle to spot `0`.

That is why spot `0` is "lost":

1. `ParkingFloor.park()` assigns the vehicle to spot `0` and increments `filled`.
2. It returns `0`.
3. `ParkingLot.park()` rejects `0` because `if spot_id:` is false.
4. No ticket is created.
5. The system now has an occupied spot with no corresponding session record.

That breaks your intended invariant that an occupied spot maps 1-to-1 with an active ticket.

How to fix it:

```python
spot_id = parking_floor.park(vehicle)
if spot_id is not None:
    ticket = ParkingTicket(spot_id=spot_id, vehicle_id=vehicle.id, entry_time=self.time, floor=idx)
    self.tickets[ticket.id] = ticket
```

Use `is not None` because `None` means failure in your current design, while `0` is a valid spot id.

2. Example invariants against your chosen representation

Chosen representation: `ParkingLot.tickets` stores full ticket history, both active and inactive, and `ticket.status` distinguishes whether a session is still live.

Example invariants:

1. For every `ticket` in `ParkingLot.tickets` with `ticket.status == Active`, the referenced floor and spot exist and that spot is currently occupied by `ticket.vehicle_id`.
2. Every occupied `ParkingSpot` corresponds to exactly one ticket in `ParkingLot.tickets` whose status is `Active` and whose `(floor, spot_id, vehicle_id)` matches that occupancy.
3. For any ticket in `ParkingLot.tickets` with status `Inactive`, that ticket cannot be used to trigger another successful `unpark` transition.
4. For every `ParkingFloor`, `filled` equals the number of spots in `parking_grid` whose `occupant is not None`.
5. No two `Active` tickets refer to the same `(floor, spot_id)` at the same time.

Why these fit your representation:

- They do not say `tickets` stores only active tickets, because you explicitly chose to retain inactive history.
- They separate active-session correctness from historical retention.
- They connect the aggregate index (`ParkingLot.tickets`) to the local state carriers (`ParkingSpot`, `ParkingFloor`).

3. Rule -> Owner -> Mutator -> Enforcement

| Rule / transition | Owner | Mutator | Enforcement point |
| --- | --- | --- | --- |
| Create a new parking session for a parked vehicle | `ParkingLot` owns the session index in `tickets` | `ParkingLot.park()` creates the `ParkingTicket` after floor allocation succeeds | `ParkingLot.park()` checks that allocation succeeded before inserting the ticket |
| Occupy a specific spot with a vehicle | `ParkingSpot` owns the occupancy field | `ParkingSpot.assign()` mutates `occupant` | `ParkingSpot.assign()` enforces compatibility and empty-before-occupy |
| Maintain floor occupancy count | `ParkingFloor` owns `filled` and local spot inventory | `ParkingFloor.park()` and `ParkingFloor.unpark()` update `filled` | those methods enforce count changes only when spot assignment/release succeeds |
| Reject stale or inactive ticket on unpark | `ParkingLot` owns ticket lifecycle status | `ParkingLot.unpark()` changes `ticket.status` from `Active` to `Inactive` | `ParkingLot.unpark()` checks `ticket.id in self.tickets` and `status == Active` before any successful close |
| Release a vehicle from a claimed spot | `ParkingSpot` owns occupancy state | `ParkingSpot.release()` sets `occupant = None` | `ParkingFloor.unpark()` resolves the spot, then `ParkingSpot.release(vehicle_id)` verifies identity before mutation |
| Preserve 1-to-1 mapping between occupied spot and active ticket | split ownership: spot state is owned by `ParkingSpot`, session index by `ParkingLot` | coordinated by `ParkingLot.park()` and `ParkingLot.unpark()` plus floor/spot helpers | aggregate-level enforcement should live in `ParkingLot` because it is the only component that sees both ticket history and cross-floor structure |

4. Explanation of owner vs mutator vs enforcement

- `Owner` means which object is the stable home of that piece of state.
- `Mutator` means which method actually performs the state change.
- `Enforcement point` means where the legality check lives that prevents invalid transitions.

Example: stale-ticket rejection.

- Owner: `ParkingLot`, because ticket lifecycle status lives in `self.tickets`.
- Mutator: `ParkingLot.unpark()`, because that is where the ticket is closed.
- Enforcement point: also `ParkingLot.unpark()`, because that is the external command boundary and it can reject stale/inactive tickets before spot mutation.

Example: spot occupancy.

- Owner: `ParkingSpot`, because `occupant` is stored there.
- Mutator: `ParkingSpot.assign()` and `ParkingSpot.release()`.
- Enforcement point: local compatibility and identity checks happen in the spot methods, but aggregate-level consistency with tickets must still be enforced by `ParkingLot`.

That is the key design split you are converging on: local objects own and mutate local state, while `ParkingLot` enforces cross-object invariants and external command legality.


In [ ]:
from enum import Enum
import uuid


class VehicleType(Enum):
    Bike = "Bike"
    Car = "Car"
    Truck = "Truck"

class Vehicle:
    def __init__(self, id = None, type: VehicleType = None, ticket = None):
        self.id = id
        self.type = type


class ParkingSpot:
    def __init__(self, type: VehicleType):
        self.occupant = None
        self.type = type

    # [2] Mutator for spot occupancy. Local enforcement point: only assign when the spot
    # is empty and the vehicle type is compatible with this spot.
    def assign(self, vehicle):
        if self.occupant is None and vehicle.type == self.type:
            self.occupant = vehicle
            return True
        else:
            return False
    
    # [5] Mutator for spot release. Local enforcement point: only release when the
    # occupant identity matches the expected vehicle_id.
    def release(self, vehicle_id) -> Vehicle:
        if self.occupant and self.occupant.id == vehicle_id:
            ret = self.occupant
            self.occupant = None
            return ret
        else:
            return None
    
    def get_occupant(self):
        return self.occupant


class TicketStatus(Enum):
    Active = "Active"
    Inactive = "Inactive"    


class ParkingTicket:
    def __init__(self, spot_id = None, vehicle_id = None, entry_time = None, floor: int = None):
        self.id = str(uuid.uuid4())
        self.spot_id = spot_id
        self.vehicle_id = vehicle_id
        self.entry_time = entry_time
        self.exit_time = entry_time
        self.floor = floor
        self.status = TicketStatus.Active


class ParkingFloor:
    def __init__(self,size):

        self.parking_grid = {spot_id: ParkingSpot(type = VehicleType.Bike if spot_id % 3 == 0 
                                                  else VehicleType.Car if spot_id % 3 == 1 
                                                  else VehicleType.Truck) for spot_id in range(size)}
        self.size = size
        self.filled = 0

    def full(self):
        return self.filled == self.size

    # [3] Mutator for floor-local occupancy count on successful park.
    # Enforcement point: only increment filled after a real spot assignment succeeds.
    def park(self, vehicle):
        if not self.full():
            parking_spot = None
            for spot_id, spot in self.parking_grid.items():
                if spot.assign(vehicle):
                    parking_spot = spot_id
                    break
            if parking_spot is not None:
                self.filled += 1
                return spot_id

    # [3] Mutator for floor-local occupancy count on successful unpark.
    # Enforcement point: only decrement filled after the matching spot release succeeds.
    def unpark(self, ticket) -> Vehicle:
        parking_spot = self.parking_grid.get(ticket.spot_id)
        if parking_spot:
            vehicle = parking_spot.release(ticket.vehicle_id)
            if vehicle:
                self.filled -= 1
                return vehicle
        return None
    

class ParkingLot:
    def __init__(self, parking_floors = 10, parking_floor_size = 10):
        self.parking_floors = [ParkingFloor(size = parking_floor_size) for _ in range(parking_floors)]
        self.tickets = {} # tracks all active and inactive tickets
        self.time = 0
        self.total_earnings = 0

    # [1] Aggregate command boundary for creating a new parking session.
    # Enforcement point: only create and insert a ticket after floor allocation succeeds.
    def park(self, vehicle):
        ticket = None
        for idx, parking_floor in enumerate(self.parking_floors):
            spot_id = parking_floor.park(vehicle)
            if spot_id is not None:
                ticket = ParkingTicket(spot_id=spot_id, vehicle_id=vehicle.id, entry_time=self.time, floor=idx)
                self.tickets[ticket.id] = ticket
                break
        return ticket

    def calculate_fees(self, ticket):
        # direct 1 per hour cost
        return ticket.exit_time - ticket.entry_time 

    def time_increment(self):
        self.time += 1 
        
    # [4] Aggregate command boundary for rejecting stale/inactive tickets and closing
    # an active session.
    # [6] Aggregate enforcement point for preserving the 1-to-1 mapping between active
    # tickets and occupied spots across floors.
    def unpark(self,ticket):
        if ticket.floor < len(self.parking_floors) and ticket.id in self.tickets and self.tickets[ticket.id].status == TicketStatus.Active:
            vehicle = self.parking_floors[ticket.floor].unpark(self.tickets[ticket.id])
            if vehicle:
                ticket.fees = self.calculate_fees(ticket)
                self.total_earnings += ticket.fees
                ticket.status = TicketStatus.Inactive
                return ticket.fees
        return None


### Contracts: why `None` is weak here, and what a stronger design looks like

1. Critique of your current contract style

Right now several methods use `None` or truthiness as the failure contract:

- `ParkingFloor.park()` returns `spot_id` or `None`
- `ParkingSpot.release()` returns `Vehicle` or `None`
- `ParkingLot.unpark()` returns `fee` or `None`

That is workable for a toy implementation, but it is a weak contract for LLD and a weak contract for production code.

Why it is weak:

1. `None` collapses different failure reasons into one shape.
   `no matching spot`, `lot full`, `stale ticket`, `wrong vehicle for spot`, and `bad floor reference` all risk becoming the same undifferentiated failure.
2. `None` encourages truthiness bugs.
   Your `spot_id == 0` bug happened because success was represented by a value that was later checked as a boolean.
3. `None` does not describe whether state changed.
   In a stateful system, a good contract should make it obvious whether the operation failed before mutation or failed after partial work.
4. `None` weakens enforcement reasoning.
   If every illegal transition just returns `None`, it becomes harder to say which guard rejected it and which invariant stayed preserved.

So the problem is not that `None` is always wrong. The problem is that `None` is too low-information for the command boundaries you are designing.

2. What a stronger production-grade contract would look like

A top 1% engineer would usually separate:

- internal helper contracts
- aggregate command contracts

Reasonable shape:

- local helpers may still return `Optional[...]` when the failure meaning is narrow and local
- aggregate command methods should usually return an explicit result type with success or failure reason

Example direction:

```python
# Augmentation [importance 6/10]: dataclasses keep the domain records compact and make
# the sample implementation focus on ownership and transitions instead of boilerplate.
from dataclasses import dataclass
from enum import Enum

class ParkFailure(Enum):
    LOT_FULL = 'LOT_FULL'
    NO_COMPATIBLE_SPOT = 'NO_COMPATIBLE_SPOT'

class UnparkFailure(Enum):
    TICKET_NOT_FOUND = 'TICKET_NOT_FOUND'
    TICKET_INACTIVE = 'TICKET_INACTIVE'
    SPOT_MISMATCH = 'SPOT_MISMATCH'

@dataclass
class ParkSuccess:
    ticket_id: str

@dataclass
class UnparkSuccess:
    fee: int
```

Conceptually, `ParkingLot.park()` becomes `Result[ParkSuccess, ParkFailure]` and `ParkingLot.unpark()` becomes `Result[UnparkSuccess, UnparkFailure]`.

3. Concrete critique of each current contract

| Method | Current contract | Critique | Better direction |
| --- | --- | --- | --- |
| `ParkingSpot.assign(vehicle)` | `bool` | Acceptable locally, because the failure space is small: occupied or incompatible. | `bool` is fine locally if caller already knows the reason space. |
| `ParkingSpot.release(vehicle_id)` | `Vehicle or None` | Reasonable locally, but only if caller treats this as a narrow helper, not as a public command result. | `Optional[Vehicle]` is acceptable as a local helper. |
| `ParkingFloor.park(vehicle)` | `spot_id or None` | Weak because `0` is a valid id and failure has multiple meanings. | Return `Optional[int]` but always check with `is not None`, or return an explicit local result. |
| `ParkingLot.park(vehicle)` | `ticket or None` | Too weak at the aggregate boundary; hides whether failure came from capacity, compatibility, or invariant rejection. | Return `ParkSuccess / ParkFailure`. |
| `ParkingLot.unpark(ticket)` | `fee or None` | Too weak at the aggregate boundary; cannot distinguish stale ticket, missing ticket, bad floor, or internal mismatch. | Return `UnparkSuccess / UnparkFailure`. |

4. What a top 1% engineer is optimizing for

A strong production engineer is optimizing for:

- explicit illegal transition handling
- debuggability of failures
- invariant preservation under change
- low ambiguity at the aggregate boundary
- ability to log, test, and reason about failure causes

So they would usually do this:

1. Keep local mutation helpers simple.
   Example: `ParkingSpot.assign()` can stay boolean.
2. Make aggregate commands explicit.
   Example: `ParkingLot.unpark(ticket_id)` returns either a successful close result or a typed rejection reason.
3. Ensure failure is rejected before mutation where possible.
   That keeps transitions atomic and invariants easier to defend.
4. Align contracts with observability.
   If the business later asks `why did unpark fail?`, the answer should already exist in the contract shape.

5. Formalization

In formal terms, your current `None` contract hides the distinction between:

- `delta` undefined because guard `Phi` rejected the event
- `delta` attempted but no legal target state was available
- `delta` failed because representation `X` was inconsistent

A stronger contract makes that separation visible.

Instead of:

```text
unpark(ticket) -> fee | None
```

the more formal shape is closer to:

```text
unpark(ticket_id) -> Result[UnparkSuccess, UnparkFailure]
```

where:

- success means `delta(s, x, e) = (s', x')` and invariants remain true
- failure means either the guard rejected the event or the transition is illegal, and state remains unchanged

That is why explicit contracts matter in LLD: they are not just API polish. They make the transition model and invariant preservation legible.

6. Practical recommendation for your notebook

For this exercise, the best tradeoff is:

- keep `ParkingSpot.assign()` and `ParkingSpot.release()` simple
- strengthen `ParkingLot.park()` and `ParkingLot.unpark()` contracts
- at minimum, stop relying on ambiguous `None` checks and use explicit `is not None`
- if you want one step more rigor, define explicit failure enums for aggregate operations

That would move your design noticeably closer to production-grade command boundaries.


### Are those really invariants or just relations? Response + formalization

Your pushback is correct. Some of the earlier examples were written as structural relations over the current state, but I did not make the distinction explicit enough.

A proper state invariant is not a time-step rule like `after park do X` or `during unpark do Y`.
A proper state invariant is a predicate over the current reachable state that must be true in every legal snapshot of the system.

So the better formal framing is:

- state invariant: a predicate `I(s, x)` that must hold for every reachable state
- transition rule: a condition on how `delta(s, x, e)` may move from one legal state to another
- enforcement point: where the code rejects an event that would violate an invariant

For your current chosen representation, these are better examples of true state invariants:

1. `filled == count(spot in parking_grid where spot.occupant is not None)` for every `ParkingFloor`.
   This is a real invariant because it must hold in every legal snapshot, not just during park/unpark.
2. For every `Active` ticket in `ParkingLot.tickets`, the referenced `(floor, spot_id)` exists and that spot's occupant has `vehicle_id == ticket.vehicle_id`.
   This is a snapshot property over the current state.
3. For every occupied spot, there exists exactly one `Active` ticket in `ParkingLot.tickets` pointing to that `(floor, spot_id)`.
   Again, this is a state property, not a transition description.
4. No two `Active` tickets point to the same `(floor, spot_id)`.
   This is a uniqueness invariant over the current session index.
5. No `Inactive` ticket is eligible for a successful unpark.
   This one sits on the edge between state and legality, but it is still expressible as a state predicate if you phrase it as: `inactive tickets are excluded from the set of legal unparkable sessions`.

What would be transition rules instead:

- `park(vehicle)` may move one spot from empty to occupied
- `unpark(ticket_id)` may move one session from active to inactive
- `unpark` with an inactive ticket must be rejected with no state change

Those are not invariants. Those are constraints on `delta`.

Formalization:

- `Sigma`: abstract state = floor occupancies + ticket statuses + lot clock + earnings
- `X`: concrete representation = `parking_floors`, `parking_grid`, `tickets`, scalar fields like `filled`
- `I`: invariants = the numbered snapshot predicates above
- `delta`: legal transitions for `park(vehicle)` and `unpark(ticket_id)`
- `Phi`: guards such as spot compatibility, active-ticket requirement, floor/spot existence

The important discipline is:

- invariants describe what must always be true about legal states
- transitions describe how you move between those legal states
- enforcement points reject events that would cause `delta` to produce a state where `I` is false

### Numbered owner / mutator / enforcement table

| # | Rule / transition | Owner | Mutator | Enforcement point |
| --- | --- | --- | --- | --- |
| 1 | Create a new parking session for a parked vehicle | `ParkingLot` owns the session index in `tickets` | `ParkingLot.park()` creates the `ParkingTicket` after floor allocation succeeds | `ParkingLot.park()` inserts the ticket only after a real allocation succeeds |
| 2 | Occupy a specific spot with a vehicle | `ParkingSpot` owns the occupancy field | `ParkingSpot.assign()` mutates `occupant` | `ParkingSpot.assign()` enforces empty-before-occupy and type compatibility |
| 3 | Maintain floor occupancy count | `ParkingFloor` owns `filled` and local spot inventory | `ParkingFloor.park()` and `ParkingFloor.unpark()` update `filled` | those methods change `filled` only after assignment/release succeeds |
| 4 | Reject stale or inactive ticket on unpark | `ParkingLot` owns ticket lifecycle status | `ParkingLot.unpark()` changes `ticket.status` from `Active` to `Inactive` | `ParkingLot.unpark()` checks ticket existence and active status before successful close |
| 5 | Release a vehicle from a claimed spot | `ParkingSpot` owns occupancy state | `ParkingSpot.release()` sets `occupant = None` | `ParkingSpot.release(vehicle_id)` verifies the identity before mutation |
| 6 | Preserve the 1-to-1 mapping between occupied spot and active ticket | split ownership: `ParkingSpot` owns occupancy, `ParkingLot` owns session index | coordinated by `ParkingLot.park()` / `ParkingLot.unpark()` plus floor/spot helpers | aggregate-level enforcement belongs in `ParkingLot` because only it sees the whole cross-floor session model |

### Trace to numbered code comments

The latest code cell now has numbered comments matching this table:

- `[1]` on `ParkingLot.park()`
- `[2]` on `ParkingSpot.assign()`
- `[3]` on `ParkingFloor.park()` and `ParkingFloor.unpark()`
- `[4]` on `ParkingLot.unpark()` for ticket lifecycle enforcement
- `[5]` on `ParkingSpot.release()`
- `[6]` on `ParkingLot.unpark()` for aggregate-level cross-object consistency

That numbering is there so you can trace one design rule from the table into the exact code location where mutation or enforcement is currently supposed to happen.


### Suggested contracts with LLD reasoning and formalization

1. Suggested contracts

For your current scoped parking-lot design, I would split contracts into two layers:

- local mutation-helper contracts
- aggregate command contracts

Suggested local contracts:

```python
ParkingSpot.assign(vehicle) -> bool
ParkingSpot.release(vehicle_id) -> Optional[Vehicle]
ParkingFloor.park(vehicle) -> Optional[int]
ParkingFloor.unpark(ticket) -> Optional[Vehicle]
```

Suggested aggregate contracts:

```python
ParkingLot.park(vehicle) -> Result[ParkSuccess, ParkFailure]
ParkingLot.unpark(ticket_id) -> Result[UnparkSuccess, UnparkFailure]
```

Possible result shapes:

```python
class ParkFailure(Enum):
    LOT_FULL = 'LOT_FULL'
    NO_COMPATIBLE_SPOT = 'NO_COMPATIBLE_SPOT'

class UnparkFailure(Enum):
    TICKET_NOT_FOUND = 'TICKET_NOT_FOUND'
    TICKET_INACTIVE = 'TICKET_INACTIVE'
    SPOT_RELEASE_FAILED = 'SPOT_RELEASE_FAILED'

@dataclass
class ParkSuccess:
    ticket_id: str
    floor: int
    spot_id: int

@dataclass
class UnparkSuccess:
    fee: int
    vehicle_id: str
```

2. Why these contracts fit LLD better

Low-level design is not mainly about method signatures. It is about defining:

- legal state
- legal transitions
- who may cause those transitions
- where illegal transitions are rejected

That is why the contract choice should follow the ownership and transition model.

Reasoning:

1. Local helpers can stay simple because they operate inside a narrow, well-understood state fragment.
   `ParkingSpot.assign()` only answers a small local question: can this spot take this vehicle right now?
2. Aggregate commands should be explicit because they are the public transition boundaries of the system.
   `ParkingLot.unpark(ticket_id)` is not just a helper. It is the command that closes a live session and must reject stale tickets safely.
3. A strong aggregate contract makes invariant-preserving rejection explicit.
   That is valuable both for interviews and for production systems.
4. Failure reasons at the aggregate boundary are part of the domain model, not just implementation detail.
   `TICKET_INACTIVE` and `TICKET_NOT_FOUND` are different business situations even if both are failures.

3. Suggested contract by class

| Class | Method | Suggested contract | Why |
| --- | --- | --- | --- |
| `ParkingSpot` | `assign(vehicle)` | `bool` | Local mutation with a small failure space. |
| `ParkingSpot` | `release(vehicle_id)` | `Optional[Vehicle]` | Local helper that returns released occupant on success. |
| `ParkingFloor` | `park(vehicle)` | `Optional[int]` | Floor-local search helper; returns spot id if found. |
| `ParkingFloor` | `unpark(ticket)` | `Optional[Vehicle]` | Floor-local release helper; does not own session validity. |
| `ParkingLot` | `park(vehicle)` | `Result[ParkSuccess, ParkFailure]` | Aggregate command that creates the session. |
| `ParkingLot` | `unpark(ticket_id)` | `Result[UnparkSuccess, UnparkFailure]` | Aggregate command that closes the session. |

4. Formalization

Under the LLD formal model:

- `E` includes events like `park(vehicle)` and `unpark(ticket_id)`
- `Sigma` is the abstract state: occupied spots, ticket statuses, current time, earnings
- `X` is the representation: `parking_floors`, `parking_grid`, `tickets`, `filled`, scalar fields
- `I` is the invariant set
- `delta` is the legal transition relation
- `Phi` is the guard layer that decides whether a transition may fire

A weak contract like:

```text
unpark(ticket) -> fee | None
```

hides too much. It does not tell you whether:

- `Phi` rejected the event before mutation
- `delta` had no legal next state
- the representation `X` was inconsistent and the transition failed internally

A stronger contract like:

```text
unpark(ticket_id) -> Result[UnparkSuccess, UnparkFailure]
```

matches the formal model better:

- success means a legal transition occurred and the next state still satisfies `I`
- failure means the event was rejected by guards or was illegal, and state should remain unchanged

5. Contract reasoning against your current design

Given your current choice that `ParkingLot.tickets` stores active and inactive history:

- `ParkingLot.unpark()` should not accept a full ticket object as the external contract
- it should accept `ticket_id`, resolve the authoritative session internally, then apply guards
- that makes stale-ticket checking an aggregate concern rather than trusting caller-provided object state

This is the key LLD reason to strengthen the contract: the aggregate root should own the authoritative identity and legality check for the transition.

6. Recommended next-step version

If you want a practical intermediate step without building a full generic `Result` type, use this compromise:

```python
ParkingLot.park(vehicle) -> tuple[Optional[ParkingTicket], Optional[ParkFailure]]
ParkingLot.unpark(ticket_id) -> tuple[Optional[int], Optional[UnparkFailure]]
```

That is less elegant than a real result object, but it already forces you to name failure causes and separate them from success.

From an LLD perspective, that would already be a meaningful upgrade over `None`-based aggregate contracts.


## 1. Findings

1. `High:` the aggregate boundary is still not authoritative enough because `ParkingLot.unpark()` accepts a full `ticket` object instead of `ticket_id`.
   - affected drill artifact: responsibilities and ownership
   - current quality `4/10`
   - why the issue matters now: stale-ticket rejection is supposed to be an aggregate concern, but the caller still supplies mutable session state instead of the lot resolving the session internally.
2. `High:` the close-session transition still computes fees from a ticket whose `exit_time` is never updated in the final code cell.
   - affected drill artifact: happy path and failure path
   - current quality `3/10`
   - why the issue matters now: `calculate_fees()` currently reads `ticket.exit_time - ticket.entry_time`, but `exit_time` remains equal to the entry timestamp, so a successful `unpark()` does not model the intended business result correctly.
3. `High:` the design is still under-specified at the artifact level even though the code improved; the final attempt still does not write explicit invariants, legal/illegal transitions, or a trace.
   - affected drill artifact: invariants
   - current quality `4/10`
   - why the issue matters now: the model is now close enough that the remaining bugs are exactly the kind that a written invariant and a trace would expose early.
4. `Medium:` requirements are more internally consistent than before, but they are still not written as a scoped drill artifact with assumptions, out-of-scope items, and edge cases.
   - affected drill artifact: requirements
   - current quality `5/10`
   - why the issue matters now: the notebook now mixes exercise scope, implementation choices, and critique advice, so it is still hard to tell which behaviors are intentional versus accidental.
5. `Medium:` the code comments show much better owner/mutator/enforcement intuition, but the state machine remains implicit.
   - affected drill artifact: state machine
   - current quality `4/10`
   - why the issue matters now: `Active -> Inactive` is present informally, but legal and illegal transitions are not written in a way that can be checked against code and future changes.
6. `Medium:` the final code fixed the `spot_id == 0` correctness bug and aligned spot types with `Bike/Car/Truck`, which is real progress.
   - affected drill artifact: progression across responsibilities and local enforcement
   - current quality `7/10`
   - why the issue matters now: the remaining gaps are no longer basic truthiness mistakes; they are now aggregate-boundary and lifecycle-modeling gaps.

## 2. Gap Matrix

| Artifact | Quality (1-10) | Main gap | Evidence | Priority (1-10) |
| --- | --- | --- | --- | --- |
| Requirements | 5 | Scope is implied across notes and code, not written as a clean drill artifact. | Notes mention fee handling and ticket history, but there is no 4 to 7 bullet requirements block with assumptions and edge cases. | 6 |
| Invariants | 4 | Invariants are discussed in later markdown, not committed as the final attempt's own artifact. | The notebook explains invariant examples, but the final code cell does not encode or list them explicitly. | 8 |
| State machine | 4 | Lifecycle remains implicit. | `TicketStatus.Active/Inactive` exists, but legal and illegal transitions are not written out. | 7 |
| Core entities | 7 | Entity set is mostly stable now, but session identity is still awkward at the boundary. | `ParkingLot`, `ParkingFloor`, `ParkingSpot`, `ParkingTicket`, and `Vehicle` are coherent, yet `unpark(ticket)` still leaks session handling to the caller. | 5 |
| Responsibilities and ownership | 4 | Aggregate authority is still incomplete. | `ParkingLot` is intended to own ticket lifecycle, but `unpark()` still trusts a caller-provided ticket object instead of resolving by id. | 9 |
| Interfaces | 5 | Public command contracts are too weak and too implicit. | Aggregate methods still return raw domain objects or `None`, and `unpark()` does not use an authoritative identifier-based command. | 7 |
| Data structures and concurrency | 6 | Local structures are reasonable, but no concurrency or atomicity notes exist. | `tickets`, `parking_floors`, and `parking_grid` are coherent, but there is no statement about atomic update expectations for spot release plus ticket close. | 4 |
| Happy path and failure path | 3 | No explicit trace, and the close path still misses `exit_time` update. | `unpark()` calculates fee without setting `exit_time`; no written success/failure sequence exists. | 9 |
| Requirement change | 2 | No explicit change analysis is present. | The notebook discusses better contracts, but does not walk through one scoped requirement change and impact. | 5 |

## 3. Revision Matrix

| Revision step | Targets | Priority (1-10) | Resolution importance (1-10) | Why before later edits |
| --- | --- | --- | --- | --- |
| 1. Rewrite `ParkingLot.unpark()` as an aggregate command on `ticket_id`, resolve the ticket internally, and reject illegal states before floor mutation. | Responsibilities and ownership, interfaces, state machine | 9 | 10 | This is the core ownership correction; later contract and trace work stays fuzzy until the aggregate boundary is authoritative. |
| 2. Fix the close-session transition to set `exit_time`, then compute fees, then mark the ticket inactive. | Happy path and failure path, invariants | 9 | 10 | The current success path is behaviorally wrong, so any critique of fees or lifecycle is downstream of this missing transition step. |
| 3. Write 3 to 5 explicit invariants against the chosen representation of full ticket history. | Invariants, responsibilities and ownership | 8 | 9 | Once the aggregate boundary is corrected, the invariants become the shortest way to defend `tickets` history, spot occupancy, and stale-ticket rejection. |
| 4. Write the lifecycle as a tiny state machine plus one happy path and one failure path. | State machine, happy path and failure path | 7 | 8 | This will expose whether the code and ownership model now agree on when mutation is allowed and what stays unchanged on rejection. |
| 5. Add one requirement-change analysis, such as differentiated pricing or reservable spots, and state what remains stable. | Requirement change, interfaces | 5 | 6 | The design is now mature enough that change analysis is useful, but it should come after the aggregate transition is fixed. |

## 4. Challenge Questions

1. Where is the single enforcement point that decides whether a ticket is still legally unparkable, and what input should that command accept?
2. What state must change atomically when a session moves from `Active` to `Inactive` in your current representation?
3. If an `unpark(ticket_id)` request is rejected because the ticket is inactive, which objects must remain unchanged?
4. What exact invariant ties `ParkingFloor.filled` to `ParkingSpot.occupant`, and where is it preserved on both `park()` and `unpark()`?
5. If you later add variable pricing by vehicle type, which existing owner should remain stable and which interface would actually need to vary?

## 5. Progression Critique

This is structurally better than the previous revision. The final attempt fixed the `spot_id == 0` bug, aligned the floor inventory with `Bike/Car/Truck`, and the numbered comments now map much more clearly to owner, mutator, and enforcement thinking. That is real progress, not just renaming.

The remaining problems are now concentrated at the aggregate boundary. The notebook's recent markdown correctly pushes toward `unpark(ticket_id)` and stronger result contracts, but the final code has not absorbed that correction yet. The evolution is therefore mixed: local modeling improved structurally, while the highest-leverage aggregate fix is still pending.

## 6. Intuition Check Matrix

| Artifact/comment | Signal | Assessment | Intuition quality (1-10) | Why |
| --- | --- | --- | --- | --- |
| Code comments `[1]` and `[4]/[6]` around `ParkingLot.park()` and `ParkingLot.unpark()` | High | Directionally correct but still not fully realized in code. | 7 | The comments correctly identify the aggregate command boundary and cross-object enforcement role, but the final `unpark(ticket)` signature still leaks authority to the caller. |
| Code comments `[2]` and `[5]` on `ParkingSpot.assign()` / `release()` | High | Correct design instinct. | 8 | These comments match the actual local ownership model: spot state is owned and mutated locally with narrow checks. |
| Code comments `[3]` on `ParkingFloor.park()` / `unpark()` | High | Correct design instinct with a narrow remaining caveat. | 8 | The `filled` update discipline is coherent, and the earlier spot-0 issue is now fixed. |
| Markdown cell `Contracts: why None is weak here...` | High | Correct design instinct. | 9 | This is the strongest recent reasoning artifact; it correctly identifies that aggregate commands need more explicit failure and authority boundaries. |
| Markdown cell `Are those really invariants or just relations?` | High | Correct design instinct. | 9 | The distinction between state invariants and transition rules is exactly the right formal correction for this notebook. |
| Final code cell overall | Medium | Improved but still partially local-patch driven. | 6 | The notebook's reasoning has outrun the code: the code absorbed the spot fix and type alignment, but not the final aggregate contract and close-transition corrections. |


### Focused critique: owner, mutator, and enforcement-point implementations and boundaries

## 1. Findings

1. `High:` the local ownership boundaries are mostly correct now, but the aggregate boundary is still not authoritative enough.
   - owner affected: `ParkingLot`
   - issue: `ParkingLot.unpark()` still accepts a full `ticket` object instead of authoritative `ticket_id` input.
   - why this matters: the aggregate root should own session identity resolution and stale-ticket rejection without trusting caller-provided object state.
2. `High:` your mutators are better localized than before.
   - owner affected: `ParkingSpot`, `ParkingFloor`
   - good part: `assign()` / `release()` mutate occupancy locally, and `ParkingFloor` mutates `filled` locally.
   - remaining gap: aggregate invariants connecting tickets to occupancy are still not defended explicitly before and after those local mutations.
3. `High:` enforcement is still too precondition-shaped and not yet invariant-shaped.
   - owner affected: `ParkingLot`
   - issue: checks like `ticket.id in self.tickets` and `status == Active` are present, but the code does not explicitly defend the full post-transition state such as `occupied spot <-> active ticket` and `filled == occupied count`.
   - why this matters: local guards reject some bad events, but they do not yet prove that the resulting global state is still legal.
4. `Medium:` `ParkingSpot` is the cleanest boundary in your current design.
   - assessment: ownership, mutation, and local enforcement are all aligned there.
   - caveat: it is still only a local boundary; it cannot enforce cross-object invariants by itself.
5. `Medium:` `ParkingFloor` is directionally correct, but `filled` remains the most fragile owned field.
   - assessment: the mutators are in the right methods.
   - caveat: because `filled` is a cached summary of occupancy, it is easy for future edits to desynchronize it unless you keep treating it as a state invariant.
6. `Medium:` your comments now show the right conceptual split more often than the code enforces it.
   - assessment: the notebook's design reasoning has overtaken the implementation.
   - why this matters: you are now at the stage where most remaining bugs are boundary-discipline bugs, not idea-discovery bugs.

## 2. Boundary Matrix

| Boundary | Owner quality | Mutator quality | Enforcement quality | Critique |
| --- | --- | --- | --- | --- |
| `ParkingSpot.occupant` | 8/10 | 8/10 | 7/10 | Strongest part of the design. Local state is owned and mutated in one place. |
| `ParkingFloor.filled` | 7/10 | 7/10 | 5/10 | Ownership and mutation placement are reasonable, but enforcement is weak because `filled` is a derived summary field. |
| `ParkingLot.tickets` | 8/10 | 7/10 | 6/10 | Good ownership decision, but enforcement is weakened by the `unpark(ticket)` boundary instead of `unpark(ticket_id)`. |
| `ParkingLot.park()` | 7/10 | 7/10 | 6/10 | Correct aggregate location for session creation, but contract is still too weak and result shape too implicit. |
| `ParkingLot.unpark()` | 6/10 | 6/10 | 5/10 | Correct intended owner boundary, but still the weakest high-leverage point because identity resolution and legality are not fully aggregate-authoritative. |

## 3. What is already good

1. Spot occupancy is owned by `ParkingSpot`, not by `ParkingLot` or `Vehicle`.
2. Floor inventory and occupancy count are kept at floor level.
3. Ticket history and ticket lifecycle are centered in `ParkingLot`.
4. Local mutators are narrower than earlier attempts, which is a real improvement in cohesion.

## 4. What is still leaking

1. Identity authority still leaks from caller into `ParkingLot.unpark()`.
2. Cross-object invariants are implied by comments, not fully enforced by the aggregate boundary.
3. Post-transition legality is still weaker than precondition checking.
4. `ParkingFloor` is still doing legitimate local mutation, but the aggregate root is not yet fully proving that those local mutations leave the global state valid.

## 5. Stronger target implementation

For this design, the strongest clean split is:

1. `ParkingLot.unpark(ticket_id)` owns command acceptance, ticket lookup, stale-ticket rejection, and session close.
2. `ParkingFloor.unpark(resolved_ticket)` owns local floor lookup and delegates spot release.
3. `ParkingSpot.release(vehicle_id)` owns the actual occupancy mutation.
4. After release succeeds, `ParkingLot` owns closing the session, setting `exit_time`, calculating fee, and marking status inactive.

That is a better owner/mutator/enforcement split because each layer does one level of authority:

- aggregate root decides whether the command is legal
- floor finds the right local container
- spot mutates local occupancy
- aggregate root finalizes the cross-object session state

## 6. Formalization

In the formal model, your current system is close to the right separation:

- local objects (`ParkingSpot`, `ParkingFloor`) implement localized pieces of `delta`
- `ParkingLot` should implement most of the guard layer `Phi` for external events
- invariant preservation should be checked at the aggregate boundary because only `ParkingLot` can see both the ticket index and the cross-floor occupancy model

So the remaining gap is not mostly about ownership anymore. It is about making the enforcement boundary fully authoritative and making the aggregate root prove that local mutations preserve the global invariants.


In [ ]:
from enum import Enum
import uuid


class VehicleType(Enum):
    Bike = "Bike"
    Car = "Car"
    Truck = "Truck"

class Vehicle:
    def __init__(self, id = None, type: VehicleType = None, ticket = None):
        self.id = id
        self.type = type


class ParkingSpot:
    def __init__(self, type: VehicleType):
        self.occupant = None
        self.type = type

    # [2] Mutator for spot occupancy. Local enforcement point: only assign when the spot
    # is empty and the vehicle type is compatible with this spot.
    def assign(self, vehicle):
        if self.occupant is None and vehicle.type == self.type:
            self.occupant = vehicle
            return True
        else:
            return False
    
    # [5] Mutator for spot release. Local enforcement point: only release when the
    # occupant identity matches the expected vehicle_id.
    def release(self, vehicle_id) -> Vehicle:
        if self.occupant and self.occupant.id == vehicle_id:
            ret = self.occupant
            self.occupant = None
            return ret
        else:
            return None
    
    def get_occupant(self):
        return self.occupant


# Augmentation [importance 9/10]: explicit ticket lifecycle state makes stale-ticket
# rejection and double-unpark illegality part of the model instead of implicit behavior.
class TicketStatus(Enum):
    Active = "Active"
    Inactive = "Inactive"    


class ParkingTicket:
    def __init__(self, spot_id = None, vehicle_id = None, entry_time = None, floor: int = None):
        self.id = str(uuid.uuid4())
        self.spot_id = spot_id
        self.vehicle_id = vehicle_id
        self.entry_time = entry_time
        self.exit_time = entry_time
        self.floor = floor
        self.status = TicketStatus.Active


class ParkingFloor:
    def __init__(self,size):

        self.parking_grid = {spot_id: ParkingSpot(type = VehicleType.Bike if spot_id % 3 == 0 
                                                  else VehicleType.Car if spot_id % 3 == 1 
                                                  else VehicleType.Truck) for spot_id in range(size)}
        self.size = size
        self.filled = 0

    def full(self):
        return self.filled == self.size

    # [3] Mutator for floor-local occupancy count on successful park.
    # Enforcement point: only increment filled after a real spot assignment succeeds.
    def park(self, vehicle):
        if not self.full():
            parking_spot = None
            for spot_id, spot in self.parking_grid.items():
                if spot.assign(vehicle):
                    parking_spot = spot_id
                    break
            if parking_spot is not None:
                self.filled += 1
                return spot_id

    # [3] Mutator for floor-local occupancy count on successful unpark.
    # Enforcement point: only decrement filled after the matching spot release succeeds.
    def unpark(self, ticket) -> Vehicle:
        parking_spot = self.parking_grid.get(ticket.spot_id)
        if parking_spot:
            vehicle = parking_spot.release(ticket.vehicle_id)
            if vehicle:
                self.filled -= 1
                return vehicle
        return None
    

class ParkingLot:
    def __init__(self, parking_floors = 10, parking_floor_size = 10):
        self.parking_floors = [ParkingFloor(size = parking_floor_size) for _ in range(parking_floors)]
        self.tickets = {} # tracks all active and inactive tickets
        self.time = 0
        self.total_earnings = 0

    # [1] Aggregate command boundary for creating a new parking session.
    # Enforcement point: only create and insert a ticket after floor allocation succeeds.
    def park(self, vehicle):
        ticket = None
        for idx, parking_floor in enumerate(self.parking_floors):
            spot_id = parking_floor.park(vehicle)
            if spot_id is not None:
                ticket = ParkingTicket(spot_id=spot_id, vehicle_id=vehicle.id, entry_time=self.time, floor=idx)
                self.tickets[ticket.id] = ticket
                break
        return ticket

    def calculate_fees(self, ticket):
        # direct 1 per hour cost
        return ticket.exit_time - ticket.entry_time 

    def time_increment(self):
        self.time += 1 
        
    # [4] Aggregate command boundary for rejecting stale/inactive tickets and closing
    # an active session.
    # [6] Aggregate enforcement point for preserving the 1-to-1 mapping between active
    # tickets and occupied spots across floors.
    def unpark(self, ticket_id):
        ticket = self.tickets.get(ticket_id)
        if ticket and ticket.floor < len(self.parking_floors) and ticket.status == TicketStatus.Active:
            vehicle = self.parking_floors[ticket.floor].unpark(ticket)
            if vehicle:
                ticket.exit_time = self.time
                ticket.fees = self.calculate_fees(ticket)
                self.total_earnings += ticket.fees
                ticket.status = TicketStatus.Inactive
                return ticket.fees
        return None


### Sample answers addressing the last Gap Matrix + editorial notes

## 1. Requirements

Sample answer:

- The system manages a parking lot with multiple floors.
- Each floor has spots for `Bike`, `Car`, and `Truck`.
- `park(vehicle)` allocates the first compatible available spot and returns a ticket.
- `unpark(ticket_id)` closes an active session, releases the occupied spot, and returns the fee.
- `ParkingLot` stores ticket history; tickets remain after close and use `status` to distinguish active vs inactive.
- Out of scope for this round: reservations, nearest-spot optimization, concurrency control, and lost-ticket recovery.

Editorial notes:

- Strong because it clearly states in-scope behavior and out-of-scope behavior.
- Strong because it commits to one allocation policy instead of leaving search vague.
- Strong because it matches your current ticket-history representation.

## 2. Invariants

Sample answer:

1. For every floor, `filled` equals the number of spots whose `occupant is not None`.
2. Every occupied spot corresponds to exactly one ticket in `ParkingLot.tickets` with `status == Active`.
3. Every `Active` ticket points to an existing floor and spot, and that spot is occupied by `ticket.vehicle_id`.
4. No two `Active` tickets refer to the same `(floor, spot_id)`.
5. An `Inactive` ticket cannot be used for a successful `unpark`.

Editorial notes:

- These are real state invariants, not workflow descriptions.
- They are written against your actual chosen representation, not an imaginary one.
- Invariants 2 and 3 together define the 1-to-1 relation from both directions.

## 3. State machine

Sample answer:

Spot lifecycle:

```text
EMPTY -> OCCUPIED
OCCUPIED -> EMPTY
```

Ticket/session lifecycle:

```text
ACTIVE -> INACTIVE
```

Illegal transitions:

```text
INACTIVE -> INACTIVE
missing ticket -> INACTIVE
EMPTY -> EMPTY via unpark
```

Editorial notes:

- Small is fine here. The important thing is that illegal transitions are explicit.
- This state machine is enough for your current scoped design.

## 4. Responsibilities and ownership

Sample answer:

| Rule / transition | Owner | Mutator | Enforcement |
| --- | --- | --- | --- |
| Create active parking session | `ParkingLot.tickets` | `ParkingLot.park()` | `ParkingLot.park()` inserts ticket only after successful allocation |
| Occupy spot | `ParkingSpot.occupant` | `ParkingSpot.assign()` | `ParkingSpot.assign()` checks empty + compatible |
| Maintain floor occupancy count | `ParkingFloor.filled` | `ParkingFloor.park()` / `ParkingFloor.unpark()` | floor methods update `filled` only after successful local mutation |
| Reject stale ticket | `ParkingLot.tickets` / `ticket.status` | `ParkingLot.unpark()` | `ParkingLot.unpark()` checks existence and active status |
| Release occupied spot | `ParkingSpot.occupant` | `ParkingSpot.release()` | `ParkingSpot.release(vehicle_id)` checks identity match |

Editorial notes:

- Strong because it separates state home, state changer, and legality checker.
- Strong because aggregate-wide rules stay at `ParkingLot`.

## 5. Interfaces / contracts

Sample answer:

```python
ParkingSpot.assign(vehicle) -> bool
ParkingSpot.release(vehicle_id) -> Optional[Vehicle]
ParkingFloor.park(vehicle) -> Optional[int]
ParkingFloor.unpark(ticket) -> Optional[Vehicle]
ParkingLot.park(vehicle) -> Result[ParkSuccess, ParkFailure]
ParkingLot.unpark(ticket_id) -> Result[UnparkSuccess, UnparkFailure]
```

Editorial notes:

- Local helpers stay simple.
- Aggregate commands carry explicit failure reasons because they own legality.

## 6. Data structures and concurrency

Sample answer:

- `ParkingLot.tickets: dict[ticket_id, ParkingTicket]` for O(1) ticket lookup.
- `ParkingLot.parking_floors: list[ParkingFloor]` because floors are iterated in order.
- `ParkingFloor.parking_grid: dict[spot_id, ParkingSpot]` for direct spot lookup during unpark.
- `park(vehicle)` is O(number_of_floors * spots_per_floor)` with naive search.
- `unpark(ticket_id)` is O(1) ticket lookup plus O(1) floor/spot resolution.
- Concurrency is out of scope for this round, but if added later, `park` and `unpark` must serialize updates to spot occupancy, `filled`, and ticket status.

Editorial notes:

- Strong because DS choices are justified by actual operations.
- Strong because it names the atomic update cluster that concurrency would need to protect.

## 7. Happy path and failure path

Sample answer:

Happy path:

```text
park(vehicle)
-> ParkingLot validates input
-> ParkingFloor finds compatible empty spot
-> ParkingSpot.assign(vehicle)
-> ParkingFloor increments filled
-> ParkingLot creates Active ticket
-> return ticket_id
```

Failure path:

```text
unpark(ticket_id)
-> ParkingLot resolves ticket
-> ticket.status == Inactive
-> reject request
-> no spot mutation
-> no filled update
-> no fee calculation
```

Editorial notes:

- This is where your enforcement-point story becomes concrete.
- A good failure trace states what remains unchanged.

## 8. Requirement change

Sample answer:

Requirement change: support vehicle-type-specific pricing.

- What stays stable: `ParkingLot` remains the aggregate root, ticket lifecycle remains `Active -> Inactive`, floor/spot ownership stays the same.
- What changes: fee calculation logic varies by vehicle type.
- Where the change lands: pricing policy or pricing strategy used by `ParkingLot.unpark()` after session close.

Editorial notes:

- Strong because it identifies stable boundaries first.
- Strong because it shows extensibility without pretending the whole design needs to be rebuilt.

## 9. Overall editorial note

The strongest version of your answer next time is not longer code. It is a tighter drill artifact set:

- one scoped requirements block
- 3 to 5 invariants
- one tiny state machine
- one owner/mutator/enforcement table
- one happy path and one failure path
- one requirement-change analysis

If you produce those cleanly first, the code almost writes itself and the remaining bugs get easier to catch.


# Overall Progression notes and findings
## 1. Findings

1. `High:` your strongest progression gain is that ownership reasoning is now much better than it was in the previous attempt, but the aggregate command boundary still lags behind that understanding.
   - progression area: responsibilities and ownership
   - current quality `5/10`
   - why the issue or gain matters now: you fixed local consistency issues and wrote much better owner/enforcement reasoning in markdown, but the final code still exposes `unpark(ticket)` instead of making `ParkingLot` the authoritative resolver of session identity.
2. `High:` the same trace and transition-completion gap keeps recurring across revisions.
   - progression area: state machine and end-to-end traces
   - current quality `4/10`
   - why the issue or gain matters now: the final code still closes a session without setting `exit_time`, which shows the lifecycle was improved conceptually but not fully walked as a success transition end to end.
3. `Medium:` you demonstrated a real retained skill in spotting and fixing representation-level correctness bugs.
   - progression area: mutation control and implementation discipline
   - current quality `7/10`
   - why the issue or gain matters now: the previous attempt lost spot `0` through truthiness and had weaker spot typing, while the final attempt fixes the `spot_id is not None` bug and aligns `Bike/Car/Truck` with the enum model.
4. `Medium:` your notes are now ahead of your code in a productive way rather than a confused way.
   - progression area: invariants and contract reasoning
   - current quality `6/10`
   - why the issue or gain matters now: the later markdown on invariants, owner/mutator/enforcement, and stronger contracts shows durable conceptual growth, but those gains are only partially reflected in the final implementation.
5. `Medium:` requirements discipline improved, but it is still not yet a stable habit.
   - progression area: scoping discipline
   - current quality `5/10`
   - why the issue or gain matters now: the final attempt no longer has the earlier enum/type mismatch, but the notebook still relies on surrounding notes instead of one explicit scoped requirements block.

## 2. Gap Matrix

| Area | Current quality (1-10) | Recurring gap | Evidence | Priority (1-10) | Recurrence (1-10) |
| --- | --- | --- | --- | --- | --- |
| Scoping discipline | 5 | Scope gets clarified in notes, not stabilized first as a drill artifact. | Cell 17 narrows the representation, but the notebook still lacks one clean requirements section for the final attempt. | 6 | 6 |
| Invariants | 6 | Invariants are understood later than they are written. | Cells 20 and 23 contain better invariant reasoning than the final code/attempt artifact itself. | 7 | 7 |
| State machine and lifecycle legality | 4 | Lifecycle is partially modeled but not fully completed in code. | `TicketStatus` exists, but final `unpark()` still omits `exit_time` update before fee calculation. | 9 | 8 |
| Entity modeling | 7 | Entity set is mostly stable, but session identity still leaks through the API boundary. | `ParkingLot`, `ParkingFloor`, `ParkingSpot`, `ParkingTicket`, `Vehicle` are coherent in cell 21. | 4 | 4 |
| Responsibilities and ownership | 5 | Aggregate authority is repeatedly understood in notes before landing in code. | Cell 18 improves ownership; cells 20, 23, and 24 sharpen it; final code still accepts `ticket` instead of `ticket_id`. | 9 | 8 |
| Interfaces and contracts | 4 | Public contracts stay weaker than the design maturity. | Cell 24 argues for `Result`-style aggregate commands, but final code still returns raw values/`None`. | 8 | 8 |
| Data structures and mutation control | 7 | Good local mutation control, but aggregate atomicity is still implicit. | `assign`, `release`, and `filled` updates are more disciplined in cell 21. | 4 | 4 |
| End-to-end traces | 3 | Missing happy/failure traces keep allowing partial fixes. | No explicit trace is written, and the final close-session flow still misses one required state update. | 8 | 9 |
| Change resilience | 3 | No explicit requirement-change analysis yet. | Later markdown discusses contracts and formalization, but not one concrete change-impact pass. | 5 | 5 |

## 3. Skills Gained Matrix

| Skill gained | Evidence of gain | Durability (1-10) | Transfer value (1-10) | Why it seems retained |
| --- | --- | --- | --- | --- |
| Distinguishing aggregate ownership from local state ownership | You moved session creation to `ParkingLot`, kept occupancy local to `ParkingSpot`, and later wrote explicit owner/mutator/enforcement reasoning. | 7 | 9 | This shows up across code and multiple later markdown cells, not only in one fix. |
| Catching truthiness and representation bugs at command boundaries | The final attempt fixes the `spot_id == 0` bug from the previous attempt. | 7 | 8 | You corrected a concrete correctness issue that directly affected invariant preservation. |
| Recognizing that invariants are snapshot properties, not workflow prose | Cell 23 explicitly distinguishes invariants from transition rules and formalizes the difference. | 8 | 9 | This is a conceptual correction likely to transfer to many other LLD problems. |
| Separating local helper contracts from aggregate command contracts | Cell 24 proposes simple local helper returns and stronger aggregate results. | 7 | 8 | The reasoning is coherent and specific, even though the final code has not fully adopted it yet. |
| Using comments to map owner, mutator, and enforcement points | Numbered comments in the final code align with the owner/enforcement table discussed later. | 6 | 7 | This is visible in more than one place and reflects a real modeling habit starting to form. |

## 4. Reinforcement Matrix

| Reinforcement step | Targets | Priority (1-10) | Resolution importance (1-10) | Why this should be trained next |
| --- | --- | --- | --- | --- |
| 1. Practice writing the full close-session transition before coding: guard, release, `exit_time`, fee, status, earnings. | State machine and lifecycle legality, end-to-end traces | 9 | 10 | Your biggest remaining weakness is partial transition completion even after the model improves. |
| 2. Force every aggregate command to take authoritative identity, not caller-owned mutable state. | Responsibilities and ownership, interfaces and contracts | 9 | 10 | This gap recurs across revisions and is the clearest place where your reasoning is ahead of your code. |
| 3. Start each next notebook revision with 3 to 5 explicit invariants and one happy/failure trace before touching code. | Invariants, end-to-end traces, scoping discipline | 8 | 9 | This is the cheapest process change that will prevent the same recurrence pattern. |
| 4. Train one requirement-change pass after each stable implementation draft. | Change resilience, interfaces and contracts | 5 | 6 | You are now strong enough conceptually that change analysis will expose brittleness early. |
| 5. Keep the owner/mutator/enforcement table, but make yourself point each row to one exact method signature. | Responsibilities and ownership, interfaces and contracts | 5 | 7 | This will tighten the link between good notes and code that actually follows them. |

## 5. Challenge Questions

1. In a different problem like elevator control, what would be the equivalent mistake to accepting `ticket` instead of `ticket_id` at the aggregate boundary?
2. If you rewrote `unpark()` to accept `ticket_id`, what exact sequence of checks and mutations should happen before and after spot release?
3. Which one of your current invariant ideas would most quickly expose the missing `exit_time` update if you had written it before coding?
4. What evidence in this notebook shows a real retained skill rather than a one-off patch?
5. If you added a reservation feature tomorrow, which current improvement would survive unchanged and which recurring gap would most likely reappear?

## 6. Progression Critique

`Structural gain:` from the previous attempt to the final attempt, you corrected the spot-0 truthiness bug and aligned floor spot typing with the enum model. That is not cosmetic; it shows better care around representation correctness.

`Structural gain:` your later markdown shows a real jump in formal thinking. The invariant-vs-transition distinction in cell 23 and the aggregate-vs-local contract split in cell 24 are strong signs that your LLD reasoning is becoming more durable.

`Still brittle:` the aggregate boundary remains the main recurring weakness. Across the progression, you repeatedly move the model toward lot-owned session authority, but the final code still does not fully trust the aggregate root to resolve identity and legality internally.

`Local patch:` the final code is cleaner locally, but the lifecycle path is still not fully closed. Missing `exit_time` before fee computation shows the success trace was not fully exercised as a state transition.

Overall, the trajectory is positive. Your notes are now ahead of your code in a useful way, which is much better than having code drift without a model. The next step is to make your pre-code artifacts force the final code to reflect the same ownership and lifecycle reasoning you already wrote down.

## 7. Intuition Check Matrix

| Artifact/comment | Signal | Assessment | Intuition quality (1-10) | Why |
| --- | --- | --- | --- | --- |
| Cell 17 note `Validation within Aggregate is crucial` | High | Correct design instinct. | 8 | This directly points at the right center of gravity for the problem and is reinforced later rather than abandoned. |
| Final code comments `[1]` to `[6]` in cell 21 | High | Directionally correct but not fully realized. | 7 | The comments correctly identify local and aggregate enforcement roles, but the public `unpark(ticket)` boundary still weakens that model. |
| Cell 20 owner/mutator/enforcement table | High | Correct design instinct. | 9 | This is one of the strongest reasoning artifacts in the notebook and maps well to stable LLD thinking. |
| Cell 23 invariant-vs-transition explanation | High | Strong retained conceptual gain. | 9 | This is the clearest evidence that your understanding is becoming more formal and transferable. |
| Cell 24 contract discussion | High | Strong instinct, partially unlanded in code. | 8 | You correctly separate local helper contracts from aggregate command contracts, but the final implementation still stops short of adopting the boundary fully. |
| Final code cell overall | Medium | Real progress, still model-to-code lag. | 6 | The final attempt is materially better than the previous one, but your best conceptual gains still outpace the implementation. |


In [ ]:
from dataclasses import dataclass
from enum import Enum
from typing import Optional
import uuid


class VehicleType(Enum):
    BIKE = "BIKE"
    CAR = "CAR"
    TRUCK = "TRUCK"


class TicketStatus(Enum):
    ACTIVE = "ACTIVE"
    INACTIVE = "INACTIVE"


# Augmentation [importance 8/10]: aggregate command failures are explicit enums instead
# of `None`, so the contract names why a command was rejected.
class ParkFailure(Enum):
    LOT_FULL = "LOT_FULL"
    NO_COMPATIBLE_SPOT = "NO_COMPATIBLE_SPOT"


class UnparkFailure(Enum):
    TICKET_NOT_FOUND = "TICKET_NOT_FOUND"
    TICKET_INACTIVE = "TICKET_INACTIVE"
    SPOT_RELEASE_FAILED = "SPOT_RELEASE_FAILED"


@dataclass(frozen=True)
class Vehicle:
    vehicle_id: str
    vehicle_type: VehicleType


# Change [importance 9/10]: the ticket now carries authoritative session state needed for
# cross-floor resolution and fee calculation; Vehicle no longer owns session state.
@dataclass
class ParkingTicket:
    ticket_id: str
    vehicle_id: str
    floor: int
    spot_id: int
    entry_time: int
    exit_time: Optional[int] = None
    status: TicketStatus = TicketStatus.ACTIVE
    fee: Optional[int] = None


# Augmentation [importance 8/10]: separate success payloads make aggregate boundaries
# explicit and avoid mixing success data with failure causes.
@dataclass(frozen=True)
class ParkSuccess:
    ticket_id: str
    floor: int
    spot_id: int


# Augmentation [importance 8/10]: result wrappers model a production-style command
# contract while staying lightweight for notebook use.
@dataclass(frozen=True)
class UnparkSuccess:
    ticket_id: str
    vehicle_id: str
    fee: int


@dataclass(frozen=True)
class ParkResult:
    success: Optional[ParkSuccess] = None
    failure: Optional[ParkFailure] = None


@dataclass(frozen=True)
class UnparkResult:
    success: Optional[UnparkSuccess] = None
    failure: Optional[UnparkFailure] = None


# Boundary [importance 9/10]: ParkingSpot remains the owner of local occupancy state.
class ParkingSpot:
    def __init__(self, supported_type: VehicleType):
        self.supported_type = supported_type
        self.occupant: Optional[Vehicle] = None

    def is_empty(self) -> bool:
        return self.occupant is None

    # Local mutator + local enforcement: narrow helper contract is sufficient here.
    def assign(self, vehicle: Vehicle) -> bool:
        if not self.is_empty():
            return False
        if vehicle.vehicle_type != self.supported_type:
            return False
        self.occupant = vehicle
        return True

    # Local mutator + local enforcement: release only if the claimed vehicle matches.
    def release(self, vehicle_id: str) -> Optional[Vehicle]:
        if self.occupant is None:
            return None
        if self.occupant.vehicle_id != vehicle_id:
            return None
        released = self.occupant
        self.occupant = None
        return released


# Boundary [importance 7/10]: ParkingFloor owns local inventory and the derived `filled`
# count, but it does not own session legality.
class ParkingFloor:
    def __init__(self, spot_types: list[VehicleType]):
        self.parking_grid: dict[int, ParkingSpot] = {
            spot_id: ParkingSpot(spot_type) for spot_id, spot_type in enumerate(spot_types)
        }
        self.filled = 0

    def full(self) -> bool:
        return self.filled == len(self.parking_grid)

    # Change [importance 8/10]: return Optional[int] and rely on `is None` checks so spot
    # id 0 remains a valid success instead of a truthiness bug.
    def park(self, vehicle: Vehicle) -> Optional[int]:
        for spot_id, spot in self.parking_grid.items():
            if spot.assign(vehicle):
                self.filled += 1
                return spot_id
        return None

    def unpark(self, ticket: ParkingTicket) -> Optional[Vehicle]:
        spot = self.parking_grid.get(ticket.spot_id)
        if spot is None:
            return None
        vehicle = spot.release(ticket.vehicle_id)
        if vehicle is None:
            return None
        self.filled -= 1
        return vehicle


# Boundary [importance 10/10]: ParkingLot is the aggregate root and authoritative command
# boundary for session creation, stale-ticket rejection, fee calculation, and ticket history.
class ParkingLot:
    def __init__(self, floors: list[ParkingFloor]):
        self.parking_floors = floors
        self.tickets: dict[str, ParkingTicket] = {}
        self.time = 0
        self.total_earnings = 0

    def tick(self, delta: int = 1) -> None:
        self.time += delta

    # Change [importance 7/10]: fee computation depends only on authoritative ticket state,
    # not on caller-owned objects or vehicle-owned ticket references.
    def calculate_fee(self, ticket: ParkingTicket) -> int:
        assert ticket.exit_time is not None
        return ticket.exit_time - ticket.entry_time

    # Change [importance 9/10]: aggregate command returns an explicit result and creates the
    # ticket only after local allocation succeeds.
    def park(self, vehicle: Vehicle) -> ParkResult:
        saw_non_full_floor = False
        for floor_index, floor in enumerate(self.parking_floors):
            if not floor.full():
                saw_non_full_floor = True
            spot_id = floor.park(vehicle)
            if spot_id is None:
                continue

            ticket = ParkingTicket(
                ticket_id=str(uuid.uuid4()),
                vehicle_id=vehicle.vehicle_id,
                floor=floor_index,
                spot_id=spot_id,
                entry_time=self.time,
            )
            self.tickets[ticket.ticket_id] = ticket
            return ParkResult(success=ParkSuccess(ticket.ticket_id, floor_index, spot_id))

        failure = ParkFailure.NO_COMPATIBLE_SPOT if saw_non_full_floor else ParkFailure.LOT_FULL
        return ParkResult(failure=failure)

    # Change [importance 10/10]: authoritative boundary now accepts `ticket_id` rather than a
    # caller-provided ticket object. This keeps identity resolution and stale-ticket checks
    # inside the aggregate root.
    def unpark(self, ticket_id: str) -> UnparkResult:
        # Aggregate enforcement: reject unknown or inactive sessions before any floor mutation.
        ticket = self.tickets.get(ticket_id)
        if ticket is None:
            return UnparkResult(failure=UnparkFailure.TICKET_NOT_FOUND)
        if ticket.status != TicketStatus.ACTIVE:
            return UnparkResult(failure=UnparkFailure.TICKET_INACTIVE)

        # Delegation boundary: once the command is authorized, delegate only the local physical
        # release to floor/spot helpers.
        floor = self.parking_floors[ticket.floor]
        vehicle = floor.unpark(ticket)
        if vehicle is None:
            return UnparkResult(failure=UnparkFailure.SPOT_RELEASE_FAILED)

        # Close-session sequence [importance 9/10]: finalize authoritative ticket state in one
        # place so history, fee, and lifecycle remain aligned.
        ticket.exit_time = self.time
        ticket.fee = self.calculate_fee(ticket)
        ticket.status = TicketStatus.INACTIVE
        self.total_earnings += ticket.fee
        return UnparkResult(success=UnparkSuccess(ticket.ticket_id, vehicle.vehicle_id, ticket.fee))


# Sample setup / usage: minimal happy path exercising park -> time -> unpark.
floors = [
    ParkingFloor([VehicleType.BIKE, VehicleType.CAR, VehicleType.TRUCK]),
    ParkingFloor([VehicleType.BIKE, VehicleType.CAR, VehicleType.TRUCK]),
]

lot = ParkingLot(floors)
car = Vehicle(vehicle_id="car-1", vehicle_type=VehicleType.CAR)

park_result = lot.park(car)
assert park_result.success is not None

lot.tick(3)
unpark_result = lot.unpark(park_result.success.ticket_id)
assert unpark_result.success is not None
assert unpark_result.success.fee == 3
print(park_result)
print(unpark_result)


### Key changes in the exemplar implementation

1. `ParkingLot.unpark()` now accepts `ticket_id` instead of a caller-provided ticket object.
   Why it matters: this makes the aggregate root the authoritative boundary for identity resolution and stale-ticket rejection.
2. Aggregate command results are explicit (`ParkResult`, `UnparkResult`) instead of returning raw values or `None` only.
   Why it matters: success and failure reasons are part of the contract, which is stronger for LLD and production cases.
3. `ParkingTicket` is the authoritative session record and carries `entry_time`, `exit_time`, `status`, and `fee`.
   Why it matters: session lifecycle and fee logic are no longer leaked into `Vehicle`.
4. `ParkingSpot` remains the local owner of occupancy state, and `ParkingFloor` remains the local owner of `filled` and spot inventory.
   Why it matters: local state ownership stays cohesive while aggregate-wide rules stay at `ParkingLot`.
5. The close-session flow is made explicit: resolve ticket, reject inactive/missing tickets, release spot, set `exit_time`, compute fee, mark inactive, update earnings.
   Why it matters: this makes the success path a real state transition instead of a partial mutation sequence.
6. The old `spot_id == 0` truthiness bug is avoided by using `Optional[int]` and explicit `is None` checks.
   Why it matters: valid spot ids are no longer lost at the aggregate command boundary.
7. The exemplar keeps your chosen scope and representation: multiple floors, Bike/Car/Truck spots, ticket history retained with `Active/Inactive` status, simple fee calculation.
   Why it matters: the exemplar is an upgrade of your attempt, not a switch to a different problem version.


### Design pattern analysis of the reference parking-lot demo and real-world implications

## 1. `get_instance()` analysis

In `awesome-low-level-design-main/solutions/python/parkinglot/parking_lot.py`, `ParkingLot.get_instance()` is using the **Singleton pattern**.

Why it is a Singleton:

- `ParkingLot` keeps a class-level `_instance`.
- `__init__` blocks second construction by raising if `_instance` already exists.
- `get_instance()` lazily initializes the object and returns the shared instance.
- `_lock` is used to make initialization safer under concurrency.

Pattern name:

- Singleton
- more specifically: lazy-initialized singleton with locking

Why the demo likely uses it:

- it wants one globally shared parking-lot object
- it wants one shared source of truth for floors, tickets, fee policy, and parking strategy
- it keeps the demo setup simple because the caller does not pass the lot object around

Tradeoff analysis:

- Good for a small demo because it reduces setup ceremony.
- Weak for testability and dependency isolation because global singleton state is harder to reset and mock.
- In a stronger LLD answer, a normal `ParkingLot()` instance is usually cleaner than a singleton unless the requirement explicitly needs process-wide uniqueness.

## 2. Why the manual `add_spot(...)` construction exists

The manual spot creation in `parking_lot_demo.py` is not really a formal GoF pattern by itself. It is closer to **manual object graph construction** or a small **composition root**.

What it is doing:

- explicitly creating floors
- explicitly creating spots
- explicitly wiring those spots into floors
- explicitly wiring floors into the lot
- explicitly setting the fee strategy

Why a demo does this:

- it makes the object relationships visible
- it avoids introducing factories/builders/config parsing too early
- it keeps the sample understandable in one file

What a more scalable variant would look like:

- a `ParkingLotBuilder`
- a factory for standard floor layouts
- config-driven lot initialization
- dependency injection instead of global singleton access

## 3. Strategy pattern analysis

The reference solution also uses the **Strategy pattern** in two places:

1. `ParkingStrategy`
   - `NearestFirstStrategy`
   - `FarthestFirstStrategy`
   - `BestFitStrategy`
2. `FeeStrategy`
   - `FlatRateFeeStrategy`
   - `VehicleBasedFeeStrategy`

Why this is Strategy:

- there is one stable context object: `ParkingLot`
- there is one family of varying algorithms: how to choose a spot, and how to calculate fees
- the caller can swap implementations without changing `ParkingLot` logic

Why this is a good fit here:

- allocation policy is a real variation point
- fee policy is also a real variation point
- those policies can change independently of lot state ownership

Where Strategy is stronger than `if/else` branching:

- if the allocation and pricing rules are expected to evolve
- if tests need to inject deterministic behavior
- if different deployments or customers choose different policies

Where Strategy would be overkill:

- if there is exactly one allocation rule forever
- if fee rules are trivial and fixed

## 4. Mermaid flow trace: singleton + manual construction

```mermaid
sequenceDiagram
    participant Demo
    participant ParkingLot
    participant Floor1
    participant Floor2

    Demo->>ParkingLot: get_instance()
    alt first call
        ParkingLot->>ParkingLot: create singleton instance
    else later call
        ParkingLot-->>Demo: return existing instance
    end

    Demo->>Floor1: create ParkingFloor(1)
    Demo->>Floor1: add_spot(F1-S1, SMALL)
    Demo->>Floor1: add_spot(F1-M1, MEDIUM)
    Demo->>Floor1: add_spot(F1-L1, LARGE)

    Demo->>Floor2: create ParkingFloor(2)
    Demo->>Floor2: add_spot(F2-M1, MEDIUM)
    Demo->>Floor2: add_spot(F2-M2, MEDIUM)

    Demo->>ParkingLot: add_floor(Floor1)
    Demo->>ParkingLot: add_floor(Floor2)
    Demo->>ParkingLot: set_fee_strategy(VehicleBasedFeeStrategy)
```

## 5. Mermaid flow trace: strategy-based parking

```mermaid
sequenceDiagram
    participant Client
    participant ParkingLot
    participant ParkingStrategy
    participant ParkingFloor
    participant ParkingSpot
    participant ParkingTicket

    Client->>ParkingLot: park_vehicle(vehicle)
    ParkingLot->>ParkingStrategy: find_spot(floors, vehicle)
    loop over floors according to strategy
        ParkingStrategy->>ParkingFloor: find_available_spot(vehicle)
        ParkingFloor->>ParkingSpot: can_fit_vehicle(vehicle) / is_available()
    end
    ParkingStrategy-->>ParkingLot: chosen spot or None
    alt spot found
        ParkingLot->>ParkingSpot: park_vehicle(vehicle)
        ParkingLot->>ParkingTicket: create ticket(vehicle, spot)
        ParkingLot->>ParkingLot: active_tickets[license] = ticket
        ParkingLot-->>Client: ticket
    else no spot found
        ParkingLot-->>Client: None
    end
```

## 6. Mermaid flow trace: strategy-based unparking and fees

```mermaid
sequenceDiagram
    participant Client
    participant ParkingLot
    participant ParkingSpot
    participant ParkingTicket
    participant FeeStrategy

    Client->>ParkingLot: unpark_vehicle(license_number)
    ParkingLot->>ParkingLot: lookup active_tickets[license_number]
    alt ticket found
        ParkingLot->>ParkingSpot: unpark_vehicle()
        ParkingLot->>ParkingTicket: set_exit_timestamp()
        ParkingLot->>FeeStrategy: calculate_fee(ticket)
        FeeStrategy-->>ParkingLot: fee
        ParkingLot-->>Client: fee
    else ticket missing
        ParkingLot-->>Client: None
    end
```

## 7. Design pattern analysis summary

| Pattern / approach | Where used | Why used | Main upside | Main downside |
| --- | --- | --- | --- | --- |
| Singleton | `ParkingLot.get_instance()` | one shared global lot instance | simple demo access, one shared state root | global state, weaker testability, harder reset/isolation |
| Strategy | `ParkingStrategy`, `FeeStrategy` | real algorithm variation points | swap behavior without changing lot core | more indirection if variation is not actually needed |
| Manual composition root | demo `add_spot`, `add_floor`, `set_fee_strategy` | make object graph explicit | transparent bootstrap and readable demo | verbose and not scalable for large/static configurations |

## 8. Real-world implications in frontier-lab AI harness engineering

These same patterns show up in AI harness and evaluation systems, but the tradeoffs become sharper.

Singleton implications:

- In an AI harness, a singleton often becomes a global registry for model clients, experiment state, prompt caches, or telemetry sinks.
- This is convenient for a small internal tool, but dangerous for parallel evaluations because state leakage between runs can silently corrupt results.
- In frontier-lab work, singleton-like global state is usually a risk when doing batch evals, multi-agent orchestration, or A/B model comparisons because one run can affect another.
- Better production alternative: explicit harness/session objects with dependency injection and run-scoped state.

Strategy implications:

- Strategy is very natural in AI harness engineering because selection policies genuinely vary.
- Examples: model routing strategy, retry strategy, sampler strategy, evaluator strategy, context-window packing strategy, scoring strategy, tool-selection strategy.
- This is one of the strongest reusable patterns in frontier-lab infrastructure because the core orchestration can stay stable while policies evolve rapidly.
- Good use: `EvaluationRunner` stays fixed while `ScoringStrategy`, `PromptSelectionStrategy`, or `TraceSamplingStrategy` changes.

Manual construction implications:

- In early harnesses, explicit construction of evaluators, datasets, tools, and models is often good because it makes experiment setup inspectable.
- But as the matrix of experiments grows, manual wiring becomes brittle and should usually move to config-driven composition or typed builders.
- In frontier labs, reproducibility matters, so explicit but declarative composition is usually better than ad hoc imperative setup hidden across scripts.

Practical frontier-lab takeaway:

- Use **Strategy** aggressively for true policy variation.
- Use **Singleton** cautiously, mainly for stateless shared resources or very carefully controlled global registries.
- Use **manual composition** early for clarity, then evolve toward config/builder/DI once experiment scale and reproducibility pressure increase.


### Resource allocation lessons from parking lot: what to learn next

Parking lot is not just a CRUD problem. It is a compact resource-allocation system.

What it teaches:

1. Resource identity versus resource eligibility.
   A spot has identity (`spot_id`) and capability (`size`, compatibility). Good allocators separate those two ideas cleanly.
2. Allocation policy versus ownership.
   The policy chooses *which* resource to allocate. Ownership decides *who may commit* that allocation.
3. Local fit versus global efficiency.
   First-fit, nearest-first, and best-fit may all succeed locally, but they have different long-term fragmentation effects on the whole lot.
4. Reservation versus commitment.
   In more advanced systems, finding a candidate resource is not the same as successfully committing it.
5. Release symmetry.
   Good allocation systems do not only model acquisition; they model release, reclamation, and stale-handle rejection equally carefully.
6. Fragmentation and future capacity.
   A greedy allocation that looks good now may make future requests impossible or more expensive.

How this generalizes:

- CPU / GPU scheduler: which machine or accelerator should a job land on?
- memory allocator: which block should satisfy a request?
- thread pool / worker queue: which worker gets the task?
- model routing in AI systems: which model or inference tier should serve the request?
- storage placement: which shard or volume should receive a write?

Important design dimensions to notice:

- compatibility constraints
- ordering policy
- fairness versus efficiency
- fragmentation cost
- reservation and rollback semantics
- stale handle / double release safety
- observability of allocation failure causes

Challenge questions

1. If you changed parking from first-fit to best-fit, what future request patterns become easier, and what new costs appear in the allocator?
2. What is the equivalent of memory fragmentation in a parking lot, and how would you measure it?
3. When does `find candidate spot` need to become a separate state from `spot committed to vehicle`?
4. If two cars arrive concurrently for the same last compatible spot, where must serialization or optimistic retry live?
5. If you added reservations, what new states and illegal transitions appear immediately?
6. Which metrics would tell you that your allocation strategy is globally bad even if every individual allocation succeeds?
7. In an AI harness, what is the parking-lot equivalent of `best-fit` versus `nearest-first` when routing requests to models?

Hints for further direction

- Try modeling `candidate -> reserved -> occupied -> released` instead of just `empty -> occupied -> empty`.
- Think about whether `best-fit` minimizes immediate waste but increases search cost.
- Ask which invariants are local and which are global when allocation spans multiple floors or zones.
- Try rewriting the problem as a scheduler problem: what changes, and what stays identical?
- Explore what data structures would support both fast allocation and low fragmentation.
- Consider how you would explain the same system if spots were GPUs, tickets were leases, and vehicles were jobs.

The bigger lesson is that parking lot is an approachable entry point into resource schedulers, lease systems, and allocator design. If you study it that way, it transfers far beyond interview parking-lot exercises.


# Designing a Parking Lot System

## Requirements
1. The parking lot should have multiple levels, each level with a certain number of parking spots.
2. The parking lot should support different types of vehicles, such as cars, motorcycles, and trucks.
3. Each parking spot should be able to accommodate a specific type of vehicle.
4. The system should assign a parking spot to a vehicle upon entry and release it when the vehicle exits.
5. The system should track the availability of parking spots and provide real-time information to customers.
6. The system should handle multiple entry and exit points and support concurrent access.

## Threading Extension

- No more ticker, the fee calculation should be strategy inserted and by raw time, perhaps by timestamp seconds// 5.

In [4]:
from dataclasses import dataclass
from enum import Enum
from typing import Optional
import uuid
import threading
import datetime

class VehicleType(Enum):
    BIKE = "BIKE"
    CAR = "CAR"
    TRUCK = "TRUCK"


class TicketStatus(Enum):
    ACTIVE = "ACTIVE"
    INACTIVE = "INACTIVE"


# Augmentation [importance 8/10]: aggregate command failures are explicit enums instead
# of `None`, so the contract names why a command was rejected.
class ParkFailure(Enum):
    LOT_FULL = "LOT_FULL"
    NO_COMPATIBLE_SPOT = "NO_COMPATIBLE_SPOT"


class UnparkFailure(Enum):
    TICKET_NOT_FOUND = "TICKET_NOT_FOUND"
    TICKET_INACTIVE = "TICKET_INACTIVE"
    SPOT_RELEASE_FAILED = "SPOT_RELEASE_FAILED"


@dataclass(frozen=True)
class Vehicle:
    vehicle_id: str
    vehicle_type: VehicleType


# Change [importance 9/10]: the ticket now carries authoritative session state needed for
# cross-floor resolution and fee calculation; Vehicle no longer owns session state.
@dataclass
class ParkingTicket:
    ticket_id: str
    vehicle_id: str
    floor: int
    spot_id: int
    entry_time: datetime.datetime
    exit_time: Optional[datetime.datetime] = None
    status: TicketStatus = TicketStatus.ACTIVE
    fee: Optional[int] = None


# Augmentation [importance 8/10]: separate success payloads make aggregate boundaries
# explicit and avoid mixing success data with failure causes.
@dataclass(frozen=True)
class ParkSuccess:
    ticket_id: str
    floor: int
    spot_id: int


# Augmentation [importance 8/10]: result wrappers model a production-style command
# contract while staying lightweight for notebook use.
@dataclass(frozen=True)
class UnparkSuccess:
    ticket_id: str
    vehicle_id: str
    fee: int


@dataclass(frozen=True)
class ParkResult:
    success: Optional[ParkSuccess] = None
    failure: Optional[ParkFailure] = None


@dataclass(frozen=True)
class UnparkResult:
    success: Optional[UnparkSuccess] = None
    failure: Optional[UnparkFailure] = None


# Boundary [importance 9/10]: ParkingSpot remains the owner of local occupancy state.
class ParkingSpot:
    def __init__(self, supported_type: VehicleType):
        self.supported_type = supported_type
        self.occupant: Optional[Vehicle] = None

    def is_empty(self) -> bool:
        return self.occupant is None

    # Local mutator + local enforcement: narrow helper contract is sufficient here.
    def assign(self, vehicle: Vehicle) -> bool:
        if not self.is_empty():
            return False
        if vehicle.vehicle_type != self.supported_type:
            return False
        self.occupant = vehicle
        return True

    # Local mutator + local enforcement: release only if the claimed vehicle matches.
    def release(self, vehicle_id: str) -> Optional[Vehicle]:
        if self.occupant is None:
            return None
        if self.occupant.vehicle_id != vehicle_id:
            return None
        released = self.occupant
        self.occupant = None
        return released


# Boundary [importance 7/10]: ParkingFloor owns local inventory and the derived `filled`
# count, but it does not own session legality.
class ParkingFloor:
    def __init__(self, spot_types: list[VehicleType]):
        self.parking_grid: dict[int, ParkingSpot] = {
            spot_id: ParkingSpot(spot_type) for spot_id, spot_type in enumerate(spot_types)
        }
        self.filled = 0

    def full(self) -> bool:
        return self.filled == len(self.parking_grid)

    # Change [importance 8/10]: return Optional[int] and rely on `is None` checks so spot
    # id 0 remains a valid success instead of a truthiness bug.
    def park(self, vehicle: Vehicle) -> Optional[int]:
        for spot_id, spot in self.parking_grid.items():
            if spot.assign(vehicle):
                self.filled += 1
                return spot_id
        return None

    def unpark(self, ticket: ParkingTicket) -> Optional[Vehicle]:
        spot = self.parking_grid.get(ticket.spot_id)
        if spot is None:
            return None
        vehicle = spot.release(ticket.vehicle_id)
        if vehicle is None:
            return None
        self.filled -= 1
        return vehicle



class FeeStrategy:
    @staticmethod
    def calculate_fee(ticket: ParkingTicket) -> int:
        raise NotImplementedError


class SimpleFeeStrategy(FeeStrategy):
    @staticmethod
    def calculate_fee(ticket: ParkingTicket) -> int:
        assert ticket.exit_time is not None
        elapsed = ticket.exit_time - ticket.entry_time
        return max(1, int(elapsed.total_seconds()))

# Boundary [importance 10/10]: ParkingLot is the aggregate root and authoritative command
# boundary for session creation, stale-ticket rejection, fee calculation, and ticket history.
class ParkingLot:
    def __init__(self, floors: list[ParkingFloor], fee_strategy: FeeStrategy):
        self.parking_floors = floors
        self.tickets: dict[str, ParkingTicket] = {}
        self.time = 0
        self.total_earnings = 0
        self.fee_strategy = fee_strategy
        self._lock = threading.Lock()

    # Change [importance 7/10]: fee computation depends only on authoritative ticket state,
    # not on caller-owned objects or vehicle-owned ticket references.
    def calculate_fee(self, ticket: ParkingTicket) -> int:
        assert ticket.exit_time is not None
        return self.fee_strategy.calculate_fee(ticket)

    # Change [importance 9/10]: aggregate command returns an explicit result and creates the
    # ticket only after local allocation succeeds.
    def park(self, vehicle: Vehicle) -> ParkResult:
        with self._lock:
            saw_non_full_floor = False
            for floor_index, floor in enumerate(self.parking_floors):
                if not floor.full():
                    saw_non_full_floor = True
                spot_id = floor.park(vehicle)
                if spot_id is None:
                    continue

                ticket = ParkingTicket(
                    ticket_id=str(uuid.uuid4()),
                    vehicle_id=vehicle.vehicle_id,
                    floor=floor_index,
                    spot_id=spot_id,
                    entry_time=datetime.datetime.now(),
                )
                self.tickets[ticket.ticket_id] = ticket
                return ParkResult(success=ParkSuccess(ticket.ticket_id, floor_index, spot_id))

            failure = ParkFailure.NO_COMPATIBLE_SPOT if saw_non_full_floor else ParkFailure.LOT_FULL
            return ParkResult(failure=failure)

    # Change [importance 10/10]: authoritative boundary now accepts `ticket_id` rather than a
    # caller-provided ticket object. This keeps identity resolution and stale-ticket checks
    # inside the aggregate root.
    def unpark(self, ticket_id: str) -> UnparkResult:
        with self._lock:
            # Aggregate enforcement: reject unknown or inactive sessions before any floor mutation.
            ticket = self.tickets.get(ticket_id)
            if ticket is None:
                return UnparkResult(failure=UnparkFailure.TICKET_NOT_FOUND)
            if ticket.status != TicketStatus.ACTIVE:
                return UnparkResult(failure=UnparkFailure.TICKET_INACTIVE)

            # Delegation boundary: once the command is authorized, delegate only the local physical
            # release to floor/spot helpers.
            floor = self.parking_floors[ticket.floor]
            vehicle = floor.unpark(ticket)
            if vehicle is None:
                return UnparkResult(failure=UnparkFailure.SPOT_RELEASE_FAILED)

            # Close-session sequence [importance 9/10]: finalize authoritative ticket state in one
            # place so history, fee, and lifecycle remain aligned.
            ticket.exit_time = datetime.datetime.now()
            ticket.fee = self.calculate_fee(ticket)
            ticket.status = TicketStatus.INACTIVE
            self.total_earnings += ticket.fee
            return UnparkResult(success=UnparkSuccess(ticket.ticket_id, vehicle.vehicle_id, ticket.fee))


# Sample setup / usage: minimal happy path exercising park -> time -> unpark.
floors = [
    ParkingFloor([VehicleType.BIKE, VehicleType.CAR, VehicleType.TRUCK]),
    ParkingFloor([VehicleType.BIKE, VehicleType.CAR, VehicleType.TRUCK]),
]

In [5]:
from concurrent.futures import ThreadPoolExecutor
from collections import Counter
import threading
import time

# Test-only fee strategy: avoids the arithmetic bug in SimpleFeeStrategy so threaded
# unpark tests can validate concurrency behavior instead of failing on fee math.



class ParkingLotRunner:
    @staticmethod
    def build_lot() -> ParkingLot:
        # Fresh lot per scenario: test runs must not share mutable occupancy or tickets.
        floors = [
            ParkingFloor([VehicleType.BIKE, VehicleType.CAR, VehicleType.TRUCK]),
            ParkingFloor([VehicleType.BIKE, VehicleType.CAR, VehicleType.TRUCK]),
        ]
        return ParkingLot(floors, SimpleFeeStrategy())

    @staticmethod
    def assert_invariants(lot: ParkingLot) -> None:
        # Snapshot-level consistency check: if locking is correct, these relations should
        # hold after every concurrent scenario completes.
        occupied = 0
        active_tickets = 0
        inactive_tickets = 0

        for floor in lot.parking_floors:
            # Floor-local invariant: derived counter must match actual occupied spots.
            floor_occupied = sum(1 for spot in floor.parking_grid.values() if spot.occupant is not None)
            assert floor.filled == floor_occupied, (
                f"filled mismatch: floor.filled={floor.filled}, actual={floor_occupied}"
            )
            occupied += floor_occupied

        for ticket in lot.tickets.values():
            if ticket.status == TicketStatus.ACTIVE:
                active_tickets += 1
                # Aggregate-level invariant: every active ticket still points to a real parked vehicle.
                spot = lot.parking_floors[ticket.floor].parking_grid[ticket.spot_id]
                assert spot.occupant is not None, "active ticket points to empty spot"
                assert spot.occupant.vehicle_id == ticket.vehicle_id, (
                    "active ticket vehicle_id does not match spot occupant"
                )
            else:
                inactive_tickets += 1

        assert occupied == active_tickets, (
            f"occupied spots ({occupied}) must equal active tickets ({active_tickets})"
        )
        assert active_tickets + inactive_tickets == len(lot.tickets)

    @staticmethod
    def run_concurrent_park_last_spot_test(workers: int = 12) -> dict:
        # Single compatible spot forces all racing threads to contend for the same resource.
        lot = ParkingLot([ParkingFloor([VehicleType.CAR])], TestFeeStrategy())
        # Barrier aligns thread start times so this is a real race, not an accidental sequence.
        barrier = threading.Barrier(workers)

        def worker(idx: int) -> ParkResult:
            vehicle = Vehicle(vehicle_id=f"car-{idx}", vehicle_type=VehicleType.CAR)
            barrier.wait()
            return lot.park(vehicle)

        with ThreadPoolExecutor(max_workers=workers) as executor:
            results = list(executor.map(worker, range(workers)))

        successes = [result for result in results if result.success is not None]
        failures = [result.failure for result in results if result.failure is not None]

        # Exactly one thread should win the last spot; all others must observe the lot as full.
        assert len(successes) == 1, f"expected 1 success, got {len(successes)}"
        assert len(failures) == workers - 1, f"expected {workers - 1} failures, got {len(failures)}"
        assert failures.count(ParkFailure.LOT_FULL) == workers - 1
        ParkingLotRunner.assert_invariants(lot)

        return {
            "test": "concurrent_park_last_spot",
            "workers": workers,
            "successes": len(successes),
            "failures": dict(Counter(failure.value for failure in failures)),
            "winning_ticket": successes[0].success.ticket_id,
        }

    @staticmethod
    def run_concurrent_unpark_same_ticket_test(workers: int = 12) -> dict:
        # Seed one active session, then let many threads try to close that same session.
        lot = ParkingLot([ParkingFloor([VehicleType.CAR])], TestFeeStrategy())
        parked = lot.park(Vehicle(vehicle_id="shared-car", vehicle_type=VehicleType.CAR))
        assert parked.success is not None
        ticket_id = parked.success.ticket_id
        barrier = threading.Barrier(workers)

        def worker(_: int) -> UnparkResult:
            barrier.wait()
            return lot.unpark(ticket_id)

        with ThreadPoolExecutor(max_workers=workers) as executor:
            results = list(executor.map(worker, range(workers)))

        successes = [result for result in results if result.success is not None]
        failures = [result.failure for result in results if result.failure is not None]

        # Only one close is legal. After that, every racing thread should see an inactive ticket.
        assert len(successes) == 1, f"expected 1 success, got {len(successes)}"
        assert len(failures) == workers - 1, f"expected {workers - 1} failures, got {len(failures)}"
        assert failures.count(UnparkFailure.TICKET_INACTIVE) == workers - 1
        assert lot.tickets[ticket_id].status == TicketStatus.INACTIVE
        ParkingLotRunner.assert_invariants(lot)

        return {
            "test": "concurrent_unpark_same_ticket",
            "workers": workers,
            "successes": len(successes),
            "failures": dict(Counter(failure.value for failure in failures)),
            "closed_ticket": ticket_id,
        }

    @staticmethod
    def run_mixed_park_unpark_stress_test(rounds: int = 20, park_workers: int = 6, unpark_workers: int = 6) -> dict:
        # Stress scenario: repeat concurrent park and concurrent same-ticket unpark cycles
        # to catch state drift that a single race may miss.
        lot = ParkingLotRunner.build_lot()
        parked_ticket_ids: list[str] = []

        for round_index in range(rounds):
            park_barrier = threading.Barrier(park_workers)

            def park_worker(idx: int) -> ParkResult:
                vehicle = Vehicle(
                    vehicle_id=f"round-{round_index}-car-{idx}",
                    vehicle_type=VehicleType.CAR,
                )
                park_barrier.wait()
                return lot.park(vehicle)

            with ThreadPoolExecutor(max_workers=park_workers) as executor:
                park_results = list(executor.map(park_worker, range(park_workers)))

            # Keep only successful ticket ids; park failures are acceptable once capacity is exhausted.
            for result in park_results:
                if result.success is not None:
                    parked_ticket_ids.append(result.success.ticket_id)

            if parked_ticket_ids:
                # Pick one live ticket and race many threads on the same unpark request.
                ticket_id = parked_ticket_ids.pop(0)
                unpark_barrier = threading.Barrier(unpark_workers)

                def unpark_worker(_: int) -> UnparkResult:
                    unpark_barrier.wait()
                    return lot.unpark(ticket_id)

                with ThreadPoolExecutor(max_workers=unpark_workers) as executor:
                    unpark_results = list(executor.map(unpark_worker, range(unpark_workers)))

                successful_unparks = [result for result in unpark_results if result.success is not None]
                failed_unparks = [result.failure for result in unpark_results if result.failure is not None]
                # Repeatedly validate that double-close is rejected under contention.
                assert len(successful_unparks) == 1
                assert failed_unparks.count(UnparkFailure.TICKET_INACTIVE) == unpark_workers - 1

            # End-of-round invariant check catches cumulative corruption in counters or session state.
            ParkingLotRunner.assert_invariants(lot)

        return {
            "test": "mixed_park_unpark_stress",
            "rounds": rounds,
            "remaining_active_tickets": sum(
                1 for ticket in lot.tickets.values() if ticket.status == TicketStatus.ACTIVE
            ),
            "total_tickets_seen": len(lot.tickets),
            "total_earnings": lot.total_earnings,
        }

    @staticmethod
    def main() -> None:
        print("Running threaded parking-lot checks...")

        # Run focused race tests first, then the broader stress scenario.
        summaries = [
            ParkingLotRunner.run_concurrent_park_last_spot_test(),
            ParkingLotRunner.run_concurrent_unpark_same_ticket_test(),
            ParkingLotRunner.run_mixed_park_unpark_stress_test(),
        ]

        for summary in summaries:
            print(summary)

        print("All threaded checks passed.")


ParkingLotRunner.main()


Running threaded parking-lot checks...
{'test': 'concurrent_park_last_spot', 'workers': 12, 'successes': 1, 'failures': {'LOT_FULL': 11}, 'winning_ticket': '36478c57-e690-4c74-844e-289f21c83d84'}
{'test': 'concurrent_unpark_same_ticket', 'workers': 12, 'successes': 1, 'failures': {'TICKET_INACTIVE': 11}, 'closed_ticket': '15e05984-e116-4fc8-b5de-257a202a6ece'}
{'test': 'mixed_park_unpark_stress', 'rounds': 20, 'remaining_active_tickets': 1, 'total_tickets_seen': 21, 'total_earnings': 20}
All threaded checks passed.


## 1. Findings

1. High: requirements are still underspecified for the threading extension; affected artifact: Requirements; current quality 6/10. The latest requirement block says the system should support concurrent access and real-time availability, but it does not state the atomicity contract for `park()` and `unpark()` or whether direct floor-level mutation is out of scope. That matters now because your concurrency story is only correct at the `ParkingLot` boundary, not as a blanket property of the whole model ([main.ipynb](/Users/yao/projects/yao-system-and-low-level-design/problems/low-level-design/Done/extend-review/parking-lot/src/main.ipynb:3592), [main.ipynb](/Users/yao/projects/yao-system-and-low-level-design/problems/low-level-design/Done/extend-review/parking-lot/src/main.ipynb:3594), [main.ipynb](/Users/yao/projects/yao-system-and-low-level-design/problems/low-level-design/Done/extend-review/parking-lot/src/main.ipynb:3778)).

2. High: one core invariant is still not written explicitly in the design cell even though the test runner checks it; affected artifact: Invariants; current quality 7/10. The current code preserves `occupied spots == active tickets` and `floor.filled == actual occupied spots`, but those are enforced only implicitly in code and test comments rather than stated up front as design invariants. This matters because the current attempt improved structurally, yet the strongest concurrency claim still depends on reading implementation details instead of a clear design statement ([main.ipynb](/Users/yao/projects/yao-system-and-low-level-design/problems/low-level-design/Done/extend-review/parking-lot/src/main.ipynb:3891), [main.ipynb](/Users/yao/projects/yao-system-and-low-level-design/problems/low-level-design/Done/extend-review/parking-lot/src/main.ipynb:3918)).

3. High: the latest attempt still has a correctness bug on the real `unpark()` success path; affected artifact: Happy path and failure path; current quality 5/10. `SimpleFeeStrategy.calculate_fee()` is invalid, and the runner had to bypass it with `TestFeeStrategy`, which means the notebook's main implementation path is not actually validated by the threaded tests. This matters now because a passing concurrency harness can hide a broken success path if the test swaps out the production behavior ([main.ipynb](/Users/yao/projects/yao-system-and-low-level-design/problems/low-level-design/Done/extend-review/parking-lot/src/main.ipynb:3763), [main.ipynb](/Users/yao/projects/yao-system-and-low-level-design/problems/low-level-design/Done/extend-review/parking-lot/src/main.ipynb:3870)).

4. Medium: responsibilities and ownership are now mostly correct, but the concurrency boundary is still narrower than the requirement wording suggests; affected artifact: Responsibilities and ownership; current quality 8/10. `ParkingLot` correctly owns session lifecycle and serializes `park()`/`unpark()` with one lock, while `ParkingFloor` and `ParkingSpot` remain unlocked local helpers. That matters because your aggregate boundary is good, but the current notebook does not clearly state that callers must not mutate floors or spots directly under concurrent access ([main.ipynb](/Users/yao/projects/yao-system-and-low-level-design/problems/low-level-design/Done/extend-review/parking-lot/src/main.ipynb:3695), [main.ipynb](/Users/yao/projects/yao-system-and-low-level-design/problems/low-level-design/Done/extend-review/parking-lot/src/main.ipynb:3724), [main.ipynb](/Users/yao/projects/yao-system-and-low-level-design/problems/low-level-design/Done/extend-review/parking-lot/src/main.ipynb:3769)).

5. Medium: the state machine is present in code but still not defended as a full legality model for concurrency; affected artifact: State machine; current quality 7/10. `TicketStatus.ACTIVE -> INACTIVE` and rejection of inactive tickets are implemented, but the latest notes do not clearly separate session lifecycle from spot lifecycle or say what remains unchanged on a rejected call. This matters because double-unpark safety is one of the main reasons the lock is justified ([main.ipynb](/Users/yao/projects/yao-system-and-low-level-design/problems/low-level-design/Done/extend-review/parking-lot/src/main.ipynb:3627), [main.ipynb](/Users/yao/projects/yao-system-and-low-level-design/problems/low-level-design/Done/extend-review/parking-lot/src/main.ipynb:3816), [main.ipynb](/Users/yao/projects/yao-system-and-low-level-design/problems/low-level-design/Done/extend-review/parking-lot/src/main.ipynb:3974)).

6. Medium: data-structure and concurrency reasoning improved substantially, but the design still chooses coarse serialization without stating the tradeoff; affected artifact: Data structures and concurrency; current quality 8/10. The single `Lock` does make the aggregate operations atomic, and the runner now proves last-spot and same-ticket races at the aggregate boundary. The missing part is the explicit decision that correctness is prioritized over throughput for this round, which is the real DS/concurrency argument here ([main.ipynb](/Users/yao/projects/yao-system-and-low-level-design/problems/low-level-design/Done/extend-review/parking-lot/src/main.ipynb:3778), [main.ipynb](/Users/yao/projects/yao-system-and-low-level-design/problems/low-level-design/Done/extend-review/parking-lot/src/main.ipynb:3924), [main.ipynb](/Users/yao/projects/yao-system-and-low-level-design/problems/low-level-design/Done/extend-review/parking-lot/src/main.ipynb:3990)).

7. Medium: the requirement-change artifact is still effectively missing from the latest attempt; affected artifact: Requirement change; current quality 3/10. `FeeStrategy` is a real variation point, but the notebook does not walk one concrete change request and explain what stays stable, what changes, and where the change lands. This matters because your current structure is finally good enough that change analysis would be meaningful rather than premature ([main.ipynb](/Users/yao/projects/yao-system-and-low-level-design/problems/low-level-design/Done/extend-review/parking-lot/src/main.ipynb:3757)).

## 2. Gap Matrix

| Artifact | Quality (1-10) | Main gap | Evidence | Priority (1-10) |
| --- | --- | --- | --- | --- |
| Requirements | 6 | Concurrent access is requested, but the atomicity boundary and scope restriction are not stated precisely. | Requirement mentions concurrent access and real-time availability, but the code only serializes aggregate operations. | 8 |
| Invariants | 7 | Key invariants are enforced in code/tests more than declared in the design. | Runner checks `filled` and active-ticket consistency, but the design cell does not foreground them as always-true claims. | 8 |
| State machine | 7 | Session legality exists in code but is not fully written as a lifecycle model with unchanged-state failure semantics. | `ACTIVE -> INACTIVE` exists, inactive rejection exists, but failure-state preservation is implicit. | 7 |
| Core entities | 8 | Entity set is mostly stable, with minor noise from unused fields like `time`. | `ParkingLot`, `ParkingFloor`, `ParkingSpot`, `ParkingTicket`, `Vehicle` are coherent. | 4 |
| Responsibilities and ownership | 8 | Aggregate ownership is correct, but the allowed mutation boundary under concurrency is underexplained. | `ParkingLot` owns session lifecycle and locking; helpers remain mutable and unlocked. | 9 |
| Interfaces | 8 | `FeeStrategy` is real, but its contract is undercut by a broken concrete implementation. | Variation point is legitimate; current `SimpleFeeStrategy` does not validate the contract. | 6 |
| Data structures and concurrency | 8 | Coarse-grained locking is acceptable, but the throughput tradeoff is unstated. | One global `Lock` plus ordered floor scan; runner validates serialization behavior. | 7 |
| Happy path and failure path | 5 | Real `unpark()` happy path is broken by fee math, and traces are mostly embodied in code/tests rather than written. | `SimpleFeeStrategy` bug; tests swap in `TestFeeStrategy`. | 9 |
| Requirement change | 3 | No explicit change-impact walkthrough is present in the latest attempt. | Strategy exists, but no artifact analyzes a change request end to end. | 5 |

## 3. Revision Matrix

| Revision step | Targets | Priority (1-10) | Resolution importance (1-10) | Why before later edits |
| --- | --- | --- | --- | --- |
| Rewrite the concurrency contract in the requirements and ownership sections as one explicit boundary statement: only `ParkingLot.park()` and `ParkingLot.unpark()` are concurrent command entry points. | Requirements, Responsibilities and ownership, Data structures and concurrency | 9 | 10 | Until the legal mutation boundary is explicit, your concurrency claim remains broader in prose than it is in code. |
| Fix the real fee path and then rerun the threaded checks against the actual production strategy or a corrected concrete strategy. | Happy path and failure path, Interfaces | 9 | 10 | The current test pass is partially insulated from the broken implementation, so correctness is still overstated. |
| Rewrite invariants as 3 to 5 explicit always-true statements, including `occupied spots == active tickets` and `floor.filled == actual occupied spots`. | Invariants, State machine | 8 | 9 | These invariants are the reason your lock and rejection behavior are meaningful; later refinements should build on them, not rediscover them from code. |
| Add one written happy path and one failure path for `park()` and `unpark(ticket_id)` with the exact unchanged state on rejection. | Happy path and failure path, State machine | 7 | 8 | Your code is now stable enough that trace validation will reveal the remaining gap cleanly rather than just surfacing earlier ownership confusion. |
| Add one concrete requirement change, such as pluggable pricing tiers or higher concurrency throughput, and explain what stays stable and where the change lands. | Requirement change, Interfaces, Data structures and concurrency | 5 | 6 | This is useful now because the core aggregate structure is mostly stable; earlier it would have been speculative. |

## 4. Challenge Questions

1. Where is the single enforcement point that guarantees a same-ticket double-unpark changes state exactly once, and which objects must remain unchanged on the losing threads?
2. If a caller invokes `ParkingFloor.unpark()` directly during concurrent traffic, which of your stated requirements still hold and which ones no longer have a clear enforcement point?
3. What exact invariant ties `ParkingLot.tickets`, `ParkingFloor.filled`, and `ParkingSpot.occupant` together after every successful `park()` and `unpark()`?
4. In your current design, what is the precise legal lifecycle for a parking session from creation to close, and what is the explicit illegal transition for a stale or already-inactive ticket?
5. If you replace the single aggregate lock with finer-grained locking later, which invariant is most at risk of breaking first and where would you enforce it?

## 5. Progression Critique

1. The progression is structural, not merely local. The latest attempt fixed the earlier highest-leverage issue by making `ParkingLot.unpark()` accept `ticket_id` and by centralizing stale-ticket rejection and lifecycle mutation inside the aggregate root.
2. The added runner cell is a real gain, not cosmetic. It finally exercises last-spot contention and same-ticket double-unpark races, which matches the threading extension instead of only claiming thread safety.
3. The remaining defects are now concentrated rather than diffuse. Earlier versions had unstable ownership and identity boundaries; the current version is mostly correct architecturally, and the main problems are an incomplete concurrency contract in prose, missing explicit invariants, and a broken fee implementation on the real happy path.
4. The new code still preserves the intended ownership better than prior attempts. `ParkingLot` now owns session acceptance, session close, and earnings mutation, while floors and spots stay local state carriers.

## 6. Intuition Check Matrix

| Artifact/comment | Signal | Assessment | Intuition quality (1-10) | Why |
| --- | --- | --- | --- | --- |
| `ParkingLot` boundary comment around session creation and stale-ticket rejection | High: correct design instinct | Correct and now realized in code. | 9 | The comment matches the implemented aggregate command boundary and is the main structural improvement in the notebook. |
| `ParkingSpot` and `ParkingFloor` boundary comments | High: correct design instinct | Mostly correct, with one remaining caveat about concurrent direct access. | 8 | The local owner split is right, but the notebook still needs to state that concurrent mutation must go through the aggregate root. |
| `TestFeeStrategy` comment about avoiding the arithmetic bug | High: honest local diagnosis | Correct but also exposes that the main implementation path is not yet trustworthy. | 8 | This is good debugging intuition, but it also proves the production fee path still needs correction before the design can be called fully correct. |
| Runner comments about barriers, same-ticket races, and invariant checks | High: correct design instinct | Strong and specific. | 9 | These comments finally connect concurrency testing to concrete invariants instead of vague threading claims. |
| Threading-extension markdown note | Medium: directionally correct but underspecified | Names a real variation point but does not define the concurrency contract. | 6 | It moves the design toward strategy-based pricing, but it does not say what must be atomic or what interface boundary is authoritative. |

## 7. Optional Deeper Model

1. State space: `X = {parking_floors, parking_grid occupancy, tickets, total_earnings}`. The latest design is strongest when you treat `tickets` and spot occupancy as one coordinated state, not as separate convenience fields.
2. Invariants: the two most load-bearing are `occupied spots == active tickets` and `floor.filled == count(occupied spots on that floor)`. Your runner now checks both, which is a good sign that the model is converging.
3. Transition relation: `park(vehicle)` may move one spot from empty to occupied and insert one active ticket; `unpark(ticket_id)` may move one active ticket to inactive, clear one occupied spot, and increase `total_earnings`; rejected `unpark(ticket_id)` must leave all of `tickets`, occupancy, and earnings unchanged.
4. Mutation authority: the latest design correctly places aggregate transition authority in `ParkingLot`, while `ParkingFloor` and `ParkingSpot` are local mutators. The remaining rigor gap is that this authority boundary is clearer in code comments than in the written requirements.


## Attempt to Answer Challenge Questions

1. Where is the single enforcement point that guarantees a same-ticket double-unpark changes state exactly once, and which objects must remain unchanged on the losing threads?

For the moment, it's a global lock on the parking lot. technically the finest granularity I could do would be per ticket lock. 


2. If a caller invokes `ParkingFloor.unpark()` directly during concurrent traffic, which of your stated requirements still hold and which ones no longer have a clear enforcement point?

the availability in real time is down to the atomicity of the thread holding of the lock and is consistent only after lock is released. 

3. What exact invariant ties `ParkingLot.tickets`, `ParkingFloor.filled`, and `ParkingSpot.occupant` together after every successful `park()` and `unpark()`?

ParkingTicket floor and spot id is the direct mapping that syncronizes the three objects.


4. In your current design, what is the precise legal lifecycle for a parking session from creation to close, and what is the explicit illegal transition for a stale or already-inactive ticket?

Ticket status is inactive and created, then parkingspot is found and allocated by parkinglot, then ticket status is active, then during unpark ticket is resolved after parkinglot releases the spot but before the end of unpark.


5. If you replace the single aggregate lock with finer-grained locking later, which invariant is most at risk of breaking first and where would you enforce it?

The fullness check since different threads might be checking the state one writing and one reading at the same time. I'd enforce it at the parkinglot level since it's a global level invariant. But for the functionality of the system, a global fullness is not required but just same granularity to same occupied check: eg per spot or per floor would have single lock and check if occupied yet.

## Grading Your Challenge-Question Responses

Overall: 5/10.

This is a meaningful improvement over your older answers because the aggregate-root instinct is now visible in several places. The main gap is precision. You often point at the right area, but stop before naming the exact guard, exact invariant, or exact unchanged state.

### 1. Same-ticket double-unpark enforcement

Score: 6/10

What is good:
- You correctly identified serialization at the `ParkingLot` level as the current control mechanism.
- You also recognized that finer-grained locking could exist later.

What is missing:
- The enforcement point is not just the lock. It is `ParkingLot.unpark(ticket_id)` under the lock, specifically the legality check that rejects non-`ACTIVE` tickets before floor mutation.
- You did not answer the second half of the question: on losing threads, spot occupancy, `filled`, ticket status, and `total_earnings` must remain unchanged.

### 2. Direct `ParkingFloor.unpark()` during concurrency

Score: 3/10

What is good:
- You were trying to reason about atomicity and visibility.

What is missing:
- The question was about ownership and enforcement, not just lock timing.
- If callers bypass `ParkingLot`, the local spot release may still happen, but stale-ticket rejection, one-to-one mapping between active tickets and occupied spots, synchronized ticket close, and earnings updates lose a clear single enforcement point.

### 3. Invariant tying tickets, `filled`, and `occupant`

Score: 4/10

What is good:
- You centered the relationship on `ticket.floor` and `ticket.spot_id`, which is the right identity linkage.

What is missing:
- A mapping is not yet an invariant.
- A precise invariant would say something like: every active ticket points to exactly one occupied spot with matching `vehicle_id`; every occupied spot has exactly one active ticket; and `filled == count(occupied spots on that floor)`.

### 4. Session lifecycle and illegal transition

Score: 4/10

What is good:
- You are reasoning in lifecycle terms instead of only code steps, which is real progress.

What is missing:
- The lifecycle is still not precise enough. `inactive and created` is contradictory as written.
- The important illegal transition was not stated sharply: `INACTIVE -> successful unpark` must be rejected, and nonexistent ticket -> successful unpark must also be rejected.

### 5. Risk under finer-grained locking

Score: 6/10

What is good:
- You correctly recognized that read/write races can break derived state.
- You also kept the enforcement intuition at the aggregate level.

What is missing:
- `fullness` is not the most load-bearing invariant here.
- The first thing most likely to break is the cross-object invariant tying active tickets to occupied spots and `filled`, because finer-grained locks split one logical transition across multiple mutable objects.

### Summary Matrix

| Question | Score (1-10) | Main gap |
| --- | --- | --- |
| Double-unpark enforcement point | 6 | You named the lock, but not the exact command boundary, guard, and unchanged state. |
| Direct floor mutation under concurrency | 3 | The answer drifted to atomicity instead of enforcement loss and boundary breakage. |
| Invariant tying tickets, `filled`, and `occupant` | 4 | You described a relationship, not a precise always-true state predicate. |
| Session lifecycle and illegal transition | 4 | The lifecycle wording is still imprecise and the illegal transition was not stated clearly. |
| Finer-grained locking risk | 6 | You recognized race risk, but not the highest-value invariant most at risk. |

### What This Says About Your Thinking

Your intuition is ahead of your formalization. You usually know the right region of the design, especially around aggregate ownership and race risk, but you still often answer with a mechanism or rough idea instead of a checkable statement.

The next bar is:
- name the exact owner
- name the exact guard
- name the exact state that changes
- name the exact state that remains unchanged on rejection

### Mini Re-challenge

Rewrite these three in one sentence each:

1. What exact state remains unchanged on a losing same-ticket `unpark()` thread?
2. Write one precise invariant relating `tickets`, `filled`, and `occupant`.
3. Write the illegal lifecycle transition for double-unpark in `STATE -> STATE` form.
